In [ ]:
# =========================
# Cell 0 — Mount Drive + Output Folder
# =========================
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os
from datetime import datetime

def ensure_drive_path(p: str) -> str:
    if p.startswith("/content/drive/"):
        return p
    if p.startswith("/MyDrive"):
        return "/content/drive" + p
    if p.startswith("MyDrive/"):
        return "/content/drive/" + p
    if p.startswith("/"):
        return "/content/drive/MyDrive" + p
    return "/content/drive/MyDrive/" + p

# @title Output Folder Name
OUTPUT_FOLDER_NAME = "sarl year 2025" # @param {type:"string"}
OUTPUT_PARENT = ensure_drive_path(f"/MyDrive/{OUTPUT_FOLDER_NAME}")
os.makedirs(OUTPUT_PARENT, exist_ok=True)

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = os.path.join(OUTPUT_PARENT, f"Webfleet_RDA_Audit_{RUN_TS}")
GLOBAL_DIR = os.path.join(OUTPUT_DIR, "02_Global")

os.makedirs(GLOBAL_DIR, exist_ok=True)

print("✅ OUTPUT_DIR:", OUTPUT_DIR)
print("✅ GLOBAL_DIR:", GLOBAL_DIR)


Mounted at /content/drive
✅ OUTPUT_DIR: /content/drive/MyDrive/sarl year 2025/Webfleet_RDA_Audit_20260512_101940
✅ GLOBAL_DIR: /content/drive/MyDrive/sarl year 2025/Webfleet_RDA_Audit_20260512_101940/02_Global


In [ ]:
# @title
# =========================
# Cell 1 — Imports + Settings
# =========================
import re
import math
import numpy as np
import pandas as pd

from google.colab import files
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Patch

try:
    import pytz
    TZ_NAME = "Europe/Zurich"
    TZ = pytz.timezone(TZ_NAME)
except Exception:
    from zoneinfo import ZoneInfo
    TZ_NAME = "Europe/Zurich"
    TZ = ZoneInfo(TZ_NAME)



pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 200)

# -------------------------
# Core buffers / rules
# -------------------------
WORK_END_BUFFER_MIN    = 30
CHECK_PRE_SHIFT        = True
PRE_SHIFT_BUFFER_MIN   = 30

# Internal buffer (scheme B: first gap between block #1 and #2)
INTERNAL_BLOCK_GAP_MIN = 180
INTERNAL_BUFFER_MIN    = 30
INTERNAL_BUF_MIN_OVERLAP_MIN   = 5
INTERNAL_BUF_MIN_OVERLAP_RATIO = 0.8

# Quality flags
MAX_REASONABLE_SPEED_KMH = 160
KM_MIN_FOR_FLAG_CONTEXT = 0.1

# Day classification (kept, but compact)
DAY_CLASS_METHOD   = "hybrid"
FULL_DAY_MINUTES   = 420
GAP_MERGE_MIN      = 10
BLOCKS_FULL        = 4
BLOCKS_HALF_MIN    = 2
FULL_SPAN_MINUTES  = 480
MIN_BLOCKS_FOR_SPAN = 2

# 61010 feature
ENABLE_61010_FEATURE = True
PRESTATION_61010_CODE = "61010"
PRESTATION_61010_BUFFER_MIN = 2  # ± minutes around each 61010 prestation

print("✅ Settings ready. TZ=", TZ)
print("✅ Timezone loaded:", TZ_NAME)

✅ Settings ready. TZ= Europe/Zurich
✅ Timezone loaded: Europe/Zurich


In [ ]:
# @title
# =========================
# Cell 2 — Upload RDA + Webfleet + Mapping + Planning
# =========================
print("Upload RDA (xlsx):")
up = files.upload()
RDA_FILE = "/content/" + next(iter(up))

print("\nUpload Webfleet (xlsx):")
up = files.upload()
WEBFLEET_FILE = "/content/" + next(iter(up))

print("\nUpload Mapping (xlsx):")
up = files.upload()
MAPPING_FILE = "/content/" + next(iter(up))

print("\nUpload Planning (xlsx):")
up = files.upload()
PLANNING_FILE = "/content/" + next(iter(up))

print("✅ Files:")
print("RDA_FILE      :", RDA_FILE)
print("WEBFLEET_FILE :", WEBFLEET_FILE)
print("MAPPING_FILE  :", MAPPING_FILE)
print("PLANNING_FILE :", PLANNING_FILE)

Upload RDA (xlsx):


Saving merged_2025_herewego_20260512_100821.xlsx to merged_2025_herewego_20260512_100821.xlsx

Upload Webfleet (xlsx):


Saving webfleet_ALL_TRIPS_2026-04-01_to_2026-04-30.csv to webfleet_ALL_TRIPS_2026-04-01_to_2026-04-30.csv

Upload Mapping (xlsx):


Saving mapping_collabs&clients&wf.xlsx to mapping_collabs&clients&wf.xlsx

Upload Planning (xlsx):


Saving planning april.xlsx to planning april.xlsx
✅ Files:
RDA_FILE      : /content/merged_2025_herewego_20260512_100821.xlsx
WEBFLEET_FILE : /content/webfleet_ALL_TRIPS_2026-04-01_to_2026-04-30.csv
MAPPING_FILE  : /content/mapping_collabs&clients&wf.xlsx
PLANNING_FILE : /content/planning april.xlsx


In [ ]:
# @title
# =========================
# Cell 3 — Helpers (robust)
# =========================
from pandas.api import types as pdt

def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def to_int_str(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    s = re.sub(r"\.0$", "", s)
    s = re.sub(r"\s+", "", s)
    # try numeric
    try:
        return str(int(float(s)))
    except Exception:
        return s if s else None

def extract_code_any(x):
    """Extract first 4–6 digit code from messy strings: '61010 - ...', 'Prestation 61010', etc."""
    if pd.isna(x):
        return np.nan
    s = str(x)
    m = re.search(r"(\d{4,6})", s)
    return m.group(1) if m else np.nan

def swiss_date(series_like):
    return pd.to_datetime(series_like, errors="coerce", dayfirst=True).dt.date

def swiss_dt(series_like):
    """
    Parse RDA Swiss local datetimes.
    Naive input is treated as Europe/Zurich local time.
    Aware input is converted to Europe/Zurich.
    """
    s = pd.to_datetime(series_like, errors="coerce", dayfirst=True)

    if TZ is None:
        return s

    try:
        if s.dt.tz is None:
            return s.dt.tz_localize(TZ_NAME, ambiguous="infer", nonexistent="shift_forward")
        return s.dt.tz_convert(TZ_NAME)
    except Exception:
        # Fallback row-by-row for messy mixed data
        def one(x):
            if pd.isna(x):
                return pd.NaT
            x = pd.Timestamp(x)
            if x.tzinfo is None:
                return x.tz_localize(TZ_NAME, ambiguous="infer", nonexistent="shift_forward")
            return x.tz_convert(TZ_NAME)
        return series_like.apply(one)


def ensure_tz(series_like: pd.Series) -> pd.Series:
    """
    Robust timezone parsing for Webfleet:
    - strings with Z/+0100/+01:00 are parsed as explicit timezone and converted to Europe/Zurich
    - strings without timezone are treated as Europe/Zurich local time
    """
    if TZ is None:
        return pd.to_datetime(series_like, errors="coerce", dayfirst=True)

    raw = series_like.copy()
    s = raw.astype(str).str.strip()

    # Detect explicit timezone at the end of string
    has_tz = s.str.contains(r"(?:Z|[+-]\d{2}:?\d{2})$", regex=True, na=False)

    out = pd.Series(pd.NaT, index=raw.index, dtype=f"datetime64[ns, {TZ_NAME}]")

    if has_tz.any():
        parsed = pd.to_datetime(raw.loc[has_tz], errors="coerce", utc=True)
        out.loc[has_tz] = parsed.dt.tz_convert(TZ_NAME)

    if (~has_tz).any():
        parsed = pd.to_datetime(raw.loc[~has_tz], errors="coerce", dayfirst=True)
        out.loc[~has_tz] = parsed.dt.tz_localize(
            TZ_NAME,
            ambiguous="infer",
            nonexistent="shift_forward"
        )

    return out

def trip_km_from_cols(df: pd.DataFrame) -> pd.Series:
    cand_km = ['distance','Distance','distance_km','mileage [km]','km','Mileage (km)','Trip distance [km]']
    cand_m  = ['distance [m]','Distance [m]','Trip distance [m]','distance_m','mileage [m]','Distance (m)','Meters']

    dcol = first_existing(df, cand_km + cand_m)
    if dcol:
        vals = pd.to_numeric(df[dcol], errors="coerce")
        header = str(dcol).lower()
        looks_meter = ('[m]' in header) or ('(m)' in header) or ('meter' in header) or (dcol in cand_m)
        q95 = np.nanquantile(vals, 0.95) if np.isfinite(vals).any() else np.nan
        if looks_meter or (pd.notna(q95) and q95 > 800):
            return (vals/1000.0).astype(float)
        return vals.astype(float)
    return pd.Series(np.nan, index=df.index, dtype=float)

def series_speed_kmh(km: pd.Series, start: pd.Series, end: pd.Series) -> pd.Series:
    dur_h = (pd.to_datetime(end) - pd.to_datetime(start)).dt.total_seconds() / 3600.0
    with np.errstate(divide="ignore", invalid="ignore"):
        spd = km / dur_h
    spd.replace([np.inf, -np.inf], np.nan, inplace=True)
    return spd

def pick_best_sheet(path: str, required_groups: list[list[str]], prefer_code: str | None = None):
    """
    Pick the best sheet by:
      - count of matched required column groups
      - bonus if prefer_code appears anywhere in sheet values
    """
    sheets = pd.read_excel(path, sheet_name=None)
    best = None
    best_score = -1
    for name, df in sheets.items():
        cols = set(df.columns.astype(str))
        score = 0
        for grp in required_groups:
            if any(c in cols for c in grp):
                score += 1
        if prefer_code is not None:
            try:
                hits = df.astype(str).apply(lambda c: c.str.contains(prefer_code, na=False)).sum().sum()
                if hits > 0:
                    score += 2
            except Exception:
                pass
        if score > best_score:
            best_score = score
            best = (name, df)
    return best

def drop_tz_excel_safe(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if c in out.columns and pdt.is_datetime64tz_dtype(out[c]):
            s = out[c]
            try:
                if TZ is not None:
                    s = s.dt.tz_convert(TZ)
            except Exception:
                pass
            out[c] = s.dt.tz_localize(None)
    return out

def to_plot_time(ts):
    """Matplotlib-friendly (tz-naive) local time."""
    if pd.isna(ts):
        return pd.NaT
    if hasattr(ts, "tz") and ts.tz is not None:
        return ts.tz_convert(TZ).tz_localize(None)
    try:
        return pd.to_datetime(ts)
    except Exception:
        return pd.NaT

def uniq_flags(series: pd.Series) -> str:
    vals = series.fillna("").astype(str)
    s = set()
    for v in vals:
        if not v or v == "nan":
            continue
        for f in v.split(","):
            f = f.strip()
            if f:
                s.add(f)
    return ",".join(sorted(s))

In [ ]:
# =========================
# Visualization Helpers (shared by HTML explorer and PDF export cells)
# =========================


def _to_local_naive(ts):
    """Strip timezone info from a timestamp, converting via TZ if set."""
    if pd.isna(ts):
        return pd.NaT
    ts = pd.Timestamp(ts)
    try:
        if ts.tzinfo is not None:
            if TZ is not None:
                return ts.tz_convert(TZ).tz_localize(None)
            return ts.tz_localize(None)
    except Exception:
        try:
            return ts.tz_localize(None)
        except Exception:
            pass
    return ts


def _fmt_hhmm(ts):
    """Format a timestamp as HH:MM string, or '-' if NaT."""
    ts = _to_local_naive(ts)
    if pd.isna(ts):
        return "-"
    return pd.Timestamp(ts).strftime("%H:%M")


def _fmt_hours(delta):
    """Format a timedelta as '2h30' or '3h', or '-' if invalid."""
    if delta is None or pd.isna(delta):
        return "-"
    total_min = int(round(delta.total_seconds() / 60.0))
    if total_min < 0:
        return "-"
    hh = total_min // 60
    mm = total_min % 60
    return f"{hh}h{mm:02d}" if mm else f"{hh}h"


def _fmt_span(s, e):
    """Format a time span as 'HH:MM -> HH:MM (Xh)' or '-' if invalid."""
    s = _to_local_naive(s)
    e = _to_local_naive(e)
    if pd.isna(s) or pd.isna(e) or e <= s:
        return "-"
    return f"{_fmt_hhmm(s)} \u2192 {_fmt_hhmm(e)} ({_fmt_hours(e - s)})"


def _safe_text(x):
    """Return string representation of x, or empty string if NaN/None."""
    if pd.isna(x):
        return ""
    return str(x)


def _safe_filename(x):
    """Sanitise x into a safe filename fragment."""
    s = _safe_text(x).strip()
    s = re.sub(r"[^\w\-. ]+", "_", s)
    s = re.sub(r"\s+", "_", s)
    return s[:180] if s else "collaborateur"


def _duration_mins(a, b):
    """Return duration in minutes between timestamps a and b, or NaN."""
    if pd.isna(a) or pd.isna(b):
        return np.nan
    return (pd.Timestamp(b) - pd.Timestamp(a)).total_seconds() / 60.0

In [ ]:
import os
import pandas as pd

# @title
# =========================
# Cell 4 — Load Excel (robust sheet selection)
# =========================
RDA_REQUIRED = [
    ["Jour","Date","date"],
    ["Début","Debut","Heure Début","Heure Debut","Start","start"],
    ["Fin","Heure fin","End","end"],
    ["Durée","Duree","duration","Minutes"],
    ["No collaborateur","No Collaborateur","Employee No","no_collaborateur","No collaborateur"],
]

WF_REQUIRED = [
    ["tripid","Trip ID","TripId"],
    ["tripmode","Trip Mode","trip_mode"],
    ["start_time","Start Time","Start time"],
    ["end_time","End Time","End time"],
    ["driverno","Driver No","driver_no"],
]

PLANNING_REQUIRED = [
    ["emp_nr"],
    ["date"],
    ["start"],
    ["end"],
    ["event_color"],
    ["client_absent"],
]

_, rda_ext = os.path.splitext(RDA_FILE)
if rda_ext.lower() == '.csv':
    RDA = pd.read_csv(RDA_FILE, sep=';')
    rda_sheet = os.path.basename(RDA_FILE)
else:
    rda_sheet, RDA = pick_best_sheet(
        RDA_FILE,
        RDA_REQUIRED,
        prefer_code=PRESTATION_61010_CODE if ENABLE_61010_FEATURE else None
    )

_, wf_ext = os.path.splitext(WEBFLEET_FILE)
if wf_ext.lower() == '.csv':
    WF = pd.read_csv(WEBFLEET_FILE, sep=',') # Changed separator from ';' to ','
    wf_sheet = os.path.basename(WEBFLEET_FILE)
else:
    wf_sheet, WF = pick_best_sheet(
        WEBFLEET_FILE,
        WF_REQUIRED,
        prefer_code=None
    )

plan_sheet, PLANNING = pick_best_sheet(
    PLANNING_FILE,
    PLANNING_REQUIRED,
    prefer_code=None
)

map_sheets = pd.read_excel(MAPPING_FILE, sheet_name=None)
MAP = map_sheets.get("Matched Collaborateurs", next(iter(map_sheets.values())))
map_sheet = "Matched Collaborateurs" if "Matched Collaborateurs" in map_sheets else next(iter(map_sheets.keys()))

print("✅ RDA sheet     :", rda_sheet, "shape:", RDA.shape)
print("✅ WF sheet      :", wf_sheet, "shape:", WF.shape)
print("✅ MAP sheet     :", map_sheet, "shape:", MAP.shape)
print("✅ PLANNING sheet:", plan_sheet, "shape:", PLANNING.shape)

✅ RDA sheet     : merged shape: (11696, 17)
✅ WF sheet      : webfleet_ALL_TRIPS_2026-04-01_to_2026-04-30.csv shape: (6068, 56)
✅ MAP sheet     : Matched Collaborateurs shape: (248, 6)
✅ PLANNING sheet: merged shape: (6766, 36)


In [ ]:
import pandas as pd

print("--- RAW PLANNING DATES ---")
display(PLANNING[['date', 'start', 'end']].head(10))
display(PLANNING[['date', 'start', 'end']].tail(10))

print("\n--- DATES PARSED WITHOUT FIX ---")
test_parse = pd.to_datetime(PLANNING['date'], errors='coerce', dayfirst=True)
display(test_parse.value_counts().sort_index())


--- RAW PLANNING DATES ---


,date,start,end
0,2026-04-01,2026-04-01 07:40:00,2026-04-01 07:50:00
1,2026-04-01,2026-04-01 17:20:00,2026-04-01 17:30:00
2,2026-04-01,2026-04-01 08:55:00,2026-04-01 09:15:00
3,2026-04-01,2026-04-01 09:30:00,2026-04-01 11:30:00
4,2026-04-01,2026-04-01 11:50:00,2026-04-01 12:20:00
5,2026-04-01,2026-04-01 18:55:00,2026-04-01 19:05:00
6,2026-04-01,2026-04-01 09:00:00,2026-04-01 09:50:00
7,2026-04-01,2026-04-01 09:45:00,2026-04-01 12:45:00
8,2026-04-01,2026-04-01 07:00:00,2026-04-01 08:00:00
9,2026-04-01,2026-04-01 18:00:00,2026-04-01 19:55:00


,date,start,end
6756,2026-04-29,2026-04-29 11:55:00,2026-04-29 12:10:00
6757,2026-04-29,2026-04-29 12:10:00,2026-04-29 12:20:00
6758,2026-04-29,2026-04-29 18:30:00,2026-04-29 18:50:00
6759,2026-04-29,2026-04-29 07:00:00,2026-04-29 07:45:00
6760,2026-04-29,2026-04-29 08:00:00,2026-04-29 08:58:00
6761,2026-04-29,2026-04-29 19:00:00,2026-04-29 19:20:00
6762,2026-04-29,2026-04-29 08:30:00,2026-04-29 09:30:00
6763,2026-04-29,2026-04-29 19:35:00,2026-04-29 20:05:00
6764,2026-04-29,2026-04-29 08:00:00,2026-04-29 08:30:00
6765,2026-04-29,2026-04-29 17:20:00,2026-04-29 17:50:00



--- DATES PARSED WITHOUT FIX ---


,count
date,
2026-01-04,212
2026-02-04,196
2026-03-04,361
2026-04-04,143
2026-05-04,135
2026-06-04,195
2026-07-04,206
2026-08-04,202
2026-09-04,194


In [ ]:
# @title
# =========================
# Cell 5 — Normalize RDA/WF + Mapping + prestation_code detection
# =========================
# ---- RDA columns ----
rda_cols = {
    "jour": first_existing(RDA, ["Jour","Date","date"]),
    "debut": first_existing(RDA, ["Début","Debut","Heure Début","Heure Debut","Start","start"]),
    "fin": first_existing(RDA, ["Fin","Heure fin","End","end", "fin"]),
    "duree": first_existing(RDA, ["Durée","Duree","duration","Minutes"]),
    "collab_name": first_existing(RDA, ["Collaborateur","collaborateur","Employee"]),
    "collab_no": first_existing(RDA, ["No collaborateur","No Collaborateur","Employee No","no_collaborateur"]),
}

missing = [k for k,v in rda_cols.items() if v is None and k in ["jour","debut","fin","duree","collab_no"]]
if missing:
    raise KeyError(f"RDA missing required columns: {missing}. Found: {list(RDA.columns)}")

rda = pd.DataFrame({
    "jour": swiss_date(RDA[rda_cols["jour"]]),
    "start": swiss_dt(RDA[rda_cols["debut"]]),
    "end": swiss_dt(RDA[rda_cols["fin"]]),
    "duree_min": pd.to_numeric(RDA[rda_cols["duree"]], errors="coerce"),
    "collab_name": (RDA[rda_cols["collab_name"]].astype(str) if rda_cols["collab_name"] else ""),
    "collab_no_sarl": RDA[rda_cols["collab_no"]].apply(to_int_str),
})

# duration fallback
mask = rda["duree_min"].isna() & rda["start"].notna() & rda["end"].notna()
rda.loc[mask, "duree_min"] = (rda.loc[mask, "end"] - rda.loc[mask, "start"]).dt.total_seconds()/60.0

# ---- Prestation column: choose by scanning ALL columns, maximizing 61010 hits ----
best_col = None
best_score = -1
best_hits = 0
for col in RDA.columns:
    ser = RDA[col].apply(extract_code_any)
    hits_61010 = int((ser.astype(str) == PRESTATION_61010_CODE).sum())
    hits_any   = int(ser.notna().sum())
    score = hits_61010 * 100000 + hits_any
    if score > best_score:
        best_score = score
        best_col = col
        best_hits = hits_61010

rda["prestation_code"] = np.nan
if best_col is not None:
    rda["prestation_code"] = RDA[best_col].apply(extract_code_any)
    print(f"✅ Prestation column chosen: '{best_col}' | 61010 hits: {best_hits}")
else:
    print("⚠️ Could not find any prestation-like codes in RDA.")

rda["rda_row_id"] = np.arange(len(rda), dtype=int)

# ---- WF columns ----
wf_cols = {
    "tripid": first_existing(WF, ["tripid","Trip ID","TripId"]),
    "tripmode": first_existing(WF, ["tripmode","Trip Mode","trip_mode"]),
    "start": first_existing(WF, ["start_time","Start Time","Start time"]),
    "end": first_existing(WF, ["end_time","End Time","End time"]),
    "driverno": first_existing(WF, ["driverno","Driver No","driver_no"]),
    "drivername": first_existing(WF, ["drivername","Driver Name","driver_name"]),
}
missing_wf = [k for k,v in wf_cols.items() if v is None and k in ["tripid","tripmode","start","end","driverno"]]
if missing_wf:
    raise KeyError(f"WF missing required columns: {missing_wf}. Found: {list(WF.columns)}")

wf = pd.DataFrame({
    "tripid": WF[wf_cols["tripid"]].astype(str),
    "tripmode": pd.to_numeric(WF[wf_cols["tripmode"]], errors="coerce").astype("Int64"),
    "start": ensure_tz(WF[wf_cols["start"]]),
    "end": ensure_tz(WF[wf_cols["end"]]),
    "driverno": WF[wf_cols["driverno"]].apply(to_int_str),
    "drivername": (WF[wf_cols["drivername"]].astype(str) if wf_cols["drivername"] else ""),
})
wf["km"] = pd.to_numeric(trip_km_from_cols(WF), errors="coerce").round(3)
wf["duration_min"] = (pd.to_datetime(wf["end"]) - pd.to_datetime(wf["start"])).dt.total_seconds()/60.0
wf["speed_kmh"] = series_speed_kmh(wf["km"], wf["start"], wf["end"])
wf["date"] = wf["start"].dt.date

# ---- Mapping ----
collab_id_col = first_existing(MAP, [
    "collaborateur-id","collab_id","Collaborateur_ID","collaborateur_id"
])
if not collab_id_col:
    raise KeyError("Mapping missing collab_id column (collaborateur-id/collab_id/Collaborateur_ID).")

map_no_sarl_col = first_existing(MAP, [
    "no-collaborateur-sarl-102","collab_no_sarl","No collaborateur","No Collaborateur"
])

map_name_sarl_col = first_existing(MAP, [
    "name-collaborateur",          # <-- new
    "collaborateur-sarl-102",
    "collab_sarl",
    "Collaborateur"
])

map_name_wf_col = first_existing(MAP, [
    "collaborateur-webfleet",
    "collab_webfleet",
    "Driver Name",
    "name-collaborateur"           # optional fallback
])

map_drv_main_col = first_existing(MAP, [
    "no-collaborateur-wf",         # <-- new
    "no-collaborateur-webfleet",
    "driverno",
    "Driver No"
])

map_df = pd.DataFrame({
    "collab_id": MAP[collab_id_col].astype(str),
    "collab_no_sarl": (MAP[map_no_sarl_col].apply(to_int_str) if map_no_sarl_col else None),
    "collab_name_sarl": (MAP[map_name_sarl_col].astype(str) if map_name_sarl_col else ""),
    "collab_name_wf": (MAP[map_name_wf_col].astype(str) if map_name_wf_col else ""),
    "driverno": (MAP[map_drv_main_col].apply(to_int_str) if map_drv_main_col else None),
}).dropna(subset=["collab_id"]).drop_duplicates()

# RDA map scanning multiple ID columns
RDA_ID_CANDIDATES = [
    "no-collaborateur-sarl-102",
    "no-collaborateur-sa-101",
    "no-collaborateur-ne-103",
    "No collaborateur",
    "No Collaborateur",
    "collab_no_sarl"
]

sarlno_to_id = {}
for col in RDA_ID_CANDIDATES:
    if col in MAP.columns:
        tmp = MAP[[collab_id_col, col]].dropna()
        for _, rr in tmp.iterrows():
            rno = to_int_str(rr[col])
            cid = rr[collab_id_col]
            if rno and pd.notna(cid):
                if str(rno) not in sarlno_to_id:
                    sarlno_to_id[str(rno)] = str(cid)

rda["collab_id"] = rda["collab_no_sarl"].map(sarlno_to_id)

# WF map scanning multiple ID columns
UO_ID_CANDIDATES = [
    "no-collaborateur-wf",         # <-- new
    "no-collaborateur-webfleet",
    "no-collaborateur-sa-101",
    "no-collaborateur-sarl-102",
    "no-collaborateur-ne-103",
    "collab_no_webfleet",
    "UO ID", "UO ID 2", "UO ID 3",
    "driverno", "Driver No"
]

driverno_to_id = {}
for col in UO_ID_CANDIDATES:
    if col in MAP.columns:
        tmp = MAP[[collab_id_col, col]].dropna()
        for _, rr in tmp.iterrows():
            dno = to_int_str(rr[col])
            cid = rr[collab_id_col]
            if dno and pd.notna(cid):
                driverno_to_id[str(dno)] = str(cid)

wf["collab_id"] = wf["driverno"].map(driverno_to_id)

# ---- Planning ----
plan_cols = {
    "emp_nr": first_existing(PLANNING, ["emp_nr"]),
    "date": first_existing(PLANNING, ["date"]),
    "start": first_existing(PLANNING, ["start"]),
    "end": first_existing(PLANNING, ["end"]),
    "duration": first_existing(PLANNING, ["duration"]),
    "event_color": first_existing(PLANNING, ["event_color"]),
    "client_absent": first_existing(PLANNING, ["client_absent"]),
    "type": first_existing(PLANNING, ["type"]),
    "note": first_existing(PLANNING, ["note"]),
    "client_nr": first_existing(PLANNING, ["client_nr"]),
}

missing_plan = [k for k, v in plan_cols.items() if v is None and k in ["emp_nr", "date", "start", "end", "event_color"]]
if missing_plan:
    raise KeyError(f"Planning missing required columns: {missing_plan}. Found: {list(PLANNING.columns)}")

from datetime import date as _date_type, datetime as _datetime_type, time as _time_type

# The planning export has appeared in mixed forms: real Excel dates, text dates,
# and ambiguous mm/dd/yyyy or dd/mm/yyyy strings. Keep real dates untouched;
# only resolve ambiguous text dates with RDA/WF overlap as evidence.
PLANNING_DATE_ORDER = "auto"          # "auto", "dayfirst", or "monthfirst"
PLANNING_DATE_TIE_BREAKER = "monthfirst"
PLANNING_DATE_MIN_YEAR = 2000
PLANNING_DATE_MAX_YEAR = 2100


def _as_local_timestamp(value):
    if pd.isna(value):
        return pd.NaT

    try:
        ts = pd.Timestamp(value)
    except Exception:
        return pd.NaT

    if pd.isna(ts):
        return pd.NaT

    try:
        if ts.tzinfo is not None:
            if TZ is not None:
                return ts.tz_convert(TZ_NAME).tz_localize(None)
            return ts.tz_localize(None)
    except Exception:
        try:
            return ts.tz_localize(None)
        except Exception:
            pass

    return ts


def _valid_planning_date(ts):
    ts = _as_local_timestamp(ts)
    if pd.isna(ts):
        return None

    if PLANNING_DATE_MIN_YEAR <= ts.year <= PLANNING_DATE_MAX_YEAR:
        return ts.date()

    return None


def _excel_or_compact_date(value):
    try:
        v = float(value)
    except Exception:
        return None

    if not np.isfinite(v):
        return None

    # Excel serial dates are usually around 45000 for 2023-2026.
    if 20000 <= v <= 80000:
        return _valid_planning_date(pd.Timestamp("1899-12-30") + pd.to_timedelta(v, unit="D"))

    # Also support compact yyyymmdd values if a CSV/export uses them.
    if float(v).is_integer():
        text = str(int(v))
        if re.fullmatch(r"20\d{6}", text):
            return _valid_planning_date(pd.to_datetime(text, format="%Y%m%d", errors="coerce"))

    return None


def _add_date_candidate(candidates, value, source, priority):
    dt = _valid_planning_date(value)
    if dt is not None:
        candidates.append((dt, source, priority))


def _try_timestamp_ymd(year, month, day):
    try:
        return pd.Timestamp(year=year, month=month, day=day)
    except ValueError:
        return pd.NaT


def _unique_date_candidates(candidates):
    best = {}
    for dt, source, priority in candidates:
        old = best.get(dt)
        if old is None or priority > old[1]:
            best[dt] = (source, priority)

    return [(dt, source, priority) for dt, (source, priority) in best.items()]


def _planning_scalar_date_candidates(value):
    candidates = []

    if pd.isna(value):
        return candidates

    # Native Excel dates read by pandas are already correct. Do not swap them.
    if isinstance(value, pd.Timestamp) or isinstance(value, _datetime_type):
        _add_date_candidate(candidates, value, "native_datetime", 500)
        return _unique_date_candidates(candidates)

    if isinstance(value, _date_type):
        _add_date_candidate(candidates, pd.Timestamp(value), "native_date", 500)
        return _unique_date_candidates(candidates)

    if isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool):
        dt = _excel_or_compact_date(value)
        if dt is not None:
            candidates.append((dt, "numeric_date", 480))
        return _unique_date_candidates(candidates)

    text = str(value).strip()
    if not text or text.lower() in {"nan", "nat", "none", "null"}:
        return candidates

    dt = _excel_or_compact_date(text)
    if dt is not None:
        candidates.append((dt, "numeric_text_date", 470))
        return _unique_date_candidates(candidates)

    # ISO-like dates are unambiguous and should not be day/month swapped.
    if re.match(r"^\d{4}[-/.]\d{1,2}[-/.]\d{1,2}(?:\s|$)", text):
        _add_date_candidate(candidates, pd.to_datetime(text, errors="coerce", yearfirst=True), "string_iso", 460)
        return _unique_date_candidates(candidates)

    m = re.match(r"^(\d{1,2})[./-](\d{1,2})[./-](\d{2,4})(?:\s.*)?$", text)
    if m:
        a = int(m.group(1))
        b = int(m.group(2))
        year = int(m.group(3))
        if year < 100:
            year += 2000

        if 1 <= a <= 31 and 1 <= b <= 31:
            if a > 12 and b <= 12:
                _add_date_candidate(candidates, _try_timestamp_ymd(year, b, a), "string_dayfirst_unambiguous", 430)
            elif b > 12 and a <= 12:
                _add_date_candidate(candidates, _try_timestamp_ymd(year, a, b), "string_monthfirst_unambiguous", 430)
            else:
                _add_date_candidate(candidates, _try_timestamp_ymd(year, b, a), "string_dayfirst_ambiguous", 300)
                _add_date_candidate(candidates, _try_timestamp_ymd(year, a, b), "string_monthfirst_ambiguous", 300)

    # Fallback for strings with month names or extra text.
    _add_date_candidate(candidates, pd.to_datetime(text, errors="coerce", dayfirst=True), "string_dayfirst_fallback", 120)
    _add_date_candidate(candidates, pd.to_datetime(text, errors="coerce", dayfirst=False), "string_monthfirst_fallback", 120)

    return _unique_date_candidates(candidates)


def _add_ref_date(ref_by_collab, ref_all, cid, value):
    if pd.isna(cid):
        return

    cands = _planning_scalar_date_candidates(value)
    if not cands:
        return

    dt = cands[0][0]
    cid = str(cid)
    ref_by_collab.setdefault(cid, set()).add(dt)
    ref_all.add(dt)


def _build_reference_dates_for_planning():
    ref_by_collab = {}
    ref_all = set()

    if "rda" in globals() and isinstance(rda, pd.DataFrame) and not rda.empty and "collab_id" in rda.columns:
        for rr in rda.dropna(subset=["collab_id"]).itertuples(index=False):
            jour_value = getattr(rr, "jour", pd.NaT)
            start_value = getattr(rr, "start", pd.NaT)
            _add_ref_date(ref_by_collab, ref_all, rr.collab_id, jour_value if pd.notna(jour_value) else start_value)

    if "wf" in globals() and isinstance(wf, pd.DataFrame) and not wf.empty and "collab_id" in wf.columns:
        for rr in wf.dropna(subset=["collab_id"]).itertuples(index=False):
            date_value = getattr(rr, "date", pd.NaT)
            start_value = getattr(rr, "start", pd.NaT)
            _add_ref_date(ref_by_collab, ref_all, rr.collab_id, date_value if pd.notna(date_value) else start_value)

    return ref_by_collab, ref_all


PLAN_REF_BY_COLLAB, PLAN_REF_ALL_DATES = _build_reference_dates_for_planning()


def _planning_date_match_score(cid, dt):
    if dt is None:
        return 0

    score = 0
    cid_text = None if pd.isna(cid) else str(cid)

    if cid_text and dt in PLAN_REF_BY_COLLAB.get(cid_text, set()):
        score += 1000

    if dt in PLAN_REF_ALL_DATES:
        score += 100

    return score


def _candidate_for_order(value, order):
    target = f"string_{order}"
    matches = [c for c in _planning_scalar_date_candidates(value) if c[1].startswith(target)]
    if not matches:
        return None

    matches = sorted(matches, key=lambda x: x[2], reverse=True)
    return matches[0][0]


def _resolve_planning_date_order(raw_dates, collab_ids):
    requested = str(PLANNING_DATE_ORDER).strip().lower()
    if requested in {"dayfirst", "monthfirst"}:
        return requested, {"dayfirst": np.nan, "monthfirst": np.nan}

    scores = {"dayfirst": 0, "monthfirst": 0}

    for value, cid in zip(raw_dates, collab_ids):
        if pd.isna(value):
            continue

        text = str(value).strip()
        if not re.match(r"^\d{1,2}[./-]\d{1,2}[./-]\d{2,4}(?:\s.*)?$", text):
            continue

        for order in ["dayfirst", "monthfirst"]:
            dt = _candidate_for_order(value, order)
            scores[order] += _planning_date_match_score(cid, dt)

    if scores["dayfirst"] > scores["monthfirst"]:
        return "dayfirst", scores

    if scores["monthfirst"] > scores["dayfirst"]:
        return "monthfirst", scores

    return PLANNING_DATE_TIE_BREAKER, scores


def _select_planning_date(value, cid=None):
    candidates = _planning_scalar_date_candidates(value)
    if not candidates:
        return pd.NaT, "unparsed"

    ranked = []
    for dt, source, priority in candidates:
        score = priority + _planning_date_match_score(cid, dt)

        # For ambiguous strings, use the auto-resolved order as a tie-breaker.
        if source.startswith(f"string_{PLANNING_DATE_ORDER_RESOLVED}"):
            score += 20

        ranked.append((score, dt, source))

    ranked.sort(key=lambda x: x[0], reverse=True)
    _, dt, source = ranked[0]
    return dt, source


def _planning_time_parts(value):
    if pd.isna(value):
        return None

    if isinstance(value, pd.Timestamp) or isinstance(value, _datetime_type):
        ts = _as_local_timestamp(value)
        if pd.notna(ts):
            return int(ts.hour), int(ts.minute), int(ts.second)

    if isinstance(value, _time_type):
        return int(value.hour), int(value.minute), int(value.second)

    if isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool):
        try:
            v = float(value)
        except Exception:
            return None

        if not np.isfinite(v):
            return None

        if 0 <= v < 1:
            total_seconds = int(round(v * 86400)) % 86400
        elif 1 <= v < 24:
            total_seconds = int(round(v * 3600)) % 86400
        elif 20000 <= v <= 80000:
            total_seconds = int(round((v % 1) * 86400)) % 86400
        else:
            return None

        return total_seconds // 3600, (total_seconds % 3600) // 60, total_seconds % 60

    text = str(value).strip()
    if not text or text.lower() in {"nan", "nat", "none", "null"}:
        return None

    m = re.search(r"(\d{1,2})[:hH](\d{2})(?::(\d{2}))?", text)
    if m:
        hh = int(m.group(1))
        mm = int(m.group(2))
        ss = int(m.group(3) or 0)
        if 0 <= hh < 24 and 0 <= mm < 60 and 0 <= ss < 60:
            return hh, mm, ss

    ts = pd.to_datetime(text, errors="coerce", dayfirst=True)
    if pd.notna(ts):
        ts = _as_local_timestamp(ts)
        return int(ts.hour), int(ts.minute), int(ts.second)

    return None


def combine_plan_date_time(parsed_dates, time_ser):
    values = []

    for d, time_value in zip(parsed_dates, time_ser):
        parts = _planning_time_parts(time_value)
        if pd.isna(d) or parts is None:
            values.append(pd.NaT)
            continue

        hh, mm, ss = parts
        values.append(pd.Timestamp.combine(d, _time_type(hh, mm, ss)))

    out = pd.to_datetime(pd.Series(values, index=time_ser.index), errors="coerce")

    if TZ is not None:
        out = out.dt.tz_localize(TZ_NAME, ambiguous="infer", nonexistent="shift_forward")

    return out


planning_emp_nr = PLANNING[plan_cols["emp_nr"]].apply(to_int_str)

plan_emp_to_id = dict(sarlno_to_id)
PLAN_EMP_ID_CANDIDATES = list(dict.fromkeys(
    RDA_ID_CANDIDATES +
    UO_ID_CANDIDATES +
    [collab_id_col, "emp_nr", "employee_nr", "Employee No"]
))

def _map_value_pairs_by_position(df, id_col, value_col):
    # MAP can contain duplicate column names; label-based row access may return a Series.
    cols = list(df.columns)
    id_positions = [i for i, c in enumerate(cols) if c == id_col]
    value_positions = [i for i, c in enumerate(cols) if c == value_col]

    for id_pos in id_positions:
        for value_pos in value_positions:
            if id_pos == value_pos:
                continue

            tmp = df.iloc[:, [id_pos, value_pos]].copy()
            tmp.columns = ["_collab_id", "_emp_value"]
            yield tmp.dropna(subset=["_collab_id", "_emp_value"])


for col in PLAN_EMP_ID_CANDIDATES:
    if col in MAP.columns and col != collab_id_col:
        for tmp in _map_value_pairs_by_position(MAP, collab_id_col, col):
            for _, rr in tmp.iterrows():
                cid = rr["_collab_id"]
                emp = to_int_str(rr["_emp_value"])
                if emp and pd.notna(cid):
                    plan_emp_to_id.setdefault(str(emp), str(cid))

for cid_pos in [i for i, c in enumerate(list(MAP.columns)) if c == collab_id_col]:
    for cid in MAP.iloc[:, cid_pos].dropna().astype(str):
        plan_emp_to_id.setdefault(cid, cid)
        cid_norm = to_int_str(cid)
        if cid_norm:
            plan_emp_to_id.setdefault(cid_norm, cid)

planning_collab_id = planning_emp_nr.map(plan_emp_to_id)
PLANNING_DATE_ORDER_RESOLVED, PLANNING_DATE_ORDER_SCORES = _resolve_planning_date_order(
    PLANNING[plan_cols["date"]],
    planning_collab_id,
)

selected_plan_dates = [
    _select_planning_date(value, cid)
    for value, cid in zip(PLANNING[plan_cols["date"]], planning_collab_id)
]
planning_date_values = pd.Series([x[0] for x in selected_plan_dates], index=PLANNING.index)
planning_date_sources = pd.Series([x[1] for x in selected_plan_dates], index=PLANNING.index)

planning = pd.DataFrame({
    "emp_nr": planning_emp_nr,
    "collab_id": planning_collab_id,
    "date": planning_date_values,
    "date_parse_source": planning_date_sources,
    "raw_date": PLANNING[plan_cols["date"]],
    "raw_start": PLANNING[plan_cols["start"]],
    "raw_end": PLANNING[plan_cols["end"]],
    "start": combine_plan_date_time(planning_date_values, PLANNING[plan_cols["start"]]),
    "end": combine_plan_date_time(planning_date_values, PLANNING[plan_cols["end"]]),
    "duration_min": (
        pd.to_numeric(PLANNING[plan_cols["duration"]], errors="coerce")
        if plan_cols["duration"] else np.nan
    ),
    "event_color": PLANNING[plan_cols["event_color"]].fillna("").astype(str).str.strip(),
    "client_absent": (
        PLANNING[plan_cols["client_absent"]].fillna("N").astype(str).str.strip().str.upper()
        if plan_cols["client_absent"] else "N"
    ),
    "type": (
        pd.to_numeric(PLANNING[plan_cols["type"]], errors="coerce")
        if plan_cols["type"] else np.nan
    ),
    "note": (
        PLANNING[plan_cols["note"]].fillna("").astype(str)
        if plan_cols["note"] else ""
    ),
    "client_nr": (
        PLANNING[plan_cols["client_nr"]].apply(to_int_str)
        if plan_cols["client_nr"] else None
    ),
})

# overnight safety
overnight = planning["start"].notna() & planning["end"].notna() & (planning["end"] < planning["start"])
planning.loc[overnight, "end"] = planning.loc[overnight, "end"] + pd.Timedelta(days=1)

# duration fallback
dur_missing = planning["duration_min"].isna() & planning["start"].notna() & planning["end"].notna()
planning.loc[dur_missing, "duration_min"] = (
    planning.loc[dur_missing, "end"] - planning.loc[dur_missing, "start"]
).dt.total_seconds() / 60.0

planning["date_only"] = planning["date"]
planning["event_color_key"] = planning["event_color"].fillna("").astype(str).str.strip().str.lower()

planning_before_filter = planning.copy()
planning_drop_reasons = []
for rr in planning_before_filter.itertuples(index=False):
    reasons = []
    if pd.isna(getattr(rr, "collab_id", pd.NA)):
        reasons.append("unmapped_emp_nr")
    if pd.isna(getattr(rr, "date", pd.NaT)):
        reasons.append("bad_date")
    if pd.isna(getattr(rr, "start", pd.NaT)):
        reasons.append("bad_start_time")
    if pd.isna(getattr(rr, "end", pd.NaT)):
        reasons.append("bad_end_time")
    if pd.notna(getattr(rr, "start", pd.NaT)) and pd.notna(getattr(rr, "end", pd.NaT)) and getattr(rr, "end") <= getattr(rr, "start"):
        reasons.append("non_positive_span")
    planning_drop_reasons.append(",".join(reasons))

planning_before_filter["drop_reason"] = planning_drop_reasons
planning_dropped_debug = planning_before_filter[planning_before_filter["drop_reason"].astype(str).str.len() > 0].copy()

planning = planning_before_filter[planning_before_filter["drop_reason"].astype(str).eq("")].copy()
planning.drop(columns=["drop_reason"], inplace=True)
planning["collab_id"] = planning["collab_id"].astype(str)

PLAN_COLOR_MAP = {
    "as": "#FFD166",                         # gold
    "inf": "#F15BB5",                        # pink
    "adv": "#9C6644",                        # brown
    "annule": "#D4A373",                     # tan
    "annul?": "#D4A373",                     # tan
    "demande d'horaire specifique": "#E9C46A",
    "demande d'horaire sp?cifique": "#E9C46A",
    "facturation (-24h)": "#7F5539",         # dark brown
    "avertir": "#FF99C8",                    # light pink
    "a ete averti - adv": "#B08968",         # mocha
    "a ?t? averti - adv": "#B08968",         # mocha
    "geplant": "#A68A64",                    # olive-brown
    "a avertir - inf": "#E6BE8A",            # buff
    "? avertir - inf": "#E6BE8A",            # buff
    "a avertir - adv": "#C9ADA7",            # rose-beige
    "? avertir - adv": "#C9ADA7",            # rose-beige
    "a ete averti - as": "#F6D6AD",          # pale sand
    "a ?t? averti - as": "#F6D6AD",          # pale sand
}

# client_absent = Y MUST override everything -> black
planning["plot_color"] = np.where(
    planning["client_absent"].eq("Y"),
    "#000000",
    planning["event_color_key"].map(PLAN_COLOR_MAP).fillna("#bdbdbd")
)

print("Planning date order resolved:", PLANNING_DATE_ORDER_RESOLVED, "scores:", PLANNING_DATE_ORDER_SCORES)
print("Planning rows read:", len(PLANNING))
print("Planning rows kept:", len(planning))
print("Planning rows dropped:", len(planning_dropped_debug))
if not planning_dropped_debug.empty:
    print("Planning drop reasons:", planning_dropped_debug["drop_reason"].value_counts().head(10).to_dict())
    print("Sample dropped planning rows are available in planning_dropped_debug")
print("Planning date parse sources:", planning_before_filter["date_parse_source"].value_counts(dropna=False).head(12).to_dict())
print("Planning collaborators matched:", planning["collab_id"].nunique())
print("Planning event_color values:", sorted(planning["event_color"].dropna().astype(str).unique().tolist()))
print("Planning absent rows:", int((planning["client_absent"] == "Y").sum()))

print("\n==== 61010 DEBUG ====")
print("Top prestation codes:", rda["prestation_code"].value_counts(dropna=False).head(12).to_dict())
print("61010 count:", int((rda["prestation_code"].astype(str) == PRESTATION_61010_CODE).sum()))
print("=====================\n")


✅ Prestation column chosen: 'No prestation' | 61010 hits: 3614
Planning date order resolved: monthfirst scores: {'dayfirst': 0, 'monthfirst': 0}
Planning rows read: 6766
Planning rows kept: 5483
Planning rows dropped: 1283
Planning drop reasons: {'unmapped_emp_nr': 1283}
Sample dropped planning rows are available in planning_dropped_debug
Planning date parse sources: {'string_iso': 6766}
Planning collaborators matched: 78
Planning event_color values: ['ADV', 'ANNULÉ', 'AS', 'Avertir Client changement', "Demande d'horaire spécifique", 'Facturation (-24h)', 'Geplant', 'INF', 'OPAS-C', 'Services/Ménages']
Planning absent rows: 50

==== 61010 DEBUG ====
Top prestation codes: {'11200': 4706, '61010': 3614, '11100': 1823, '14200': 337, '95900': 306, nan: 215, '16011': 154, '16012': 114, '9001': 92, '50002': 44, '11000': 44, '50020': 39}
61010 count: 3614



In [ ]:
# @title
# =========================
# Cell 6 — rda_daily (day_class + internal buffer B + outer buffers) — TZ safe
# =========================

# -------------------------------------------------------------------
# Safety: make sure Zurich timezone exists even if Cell 1 had a typo
# -------------------------------------------------------------------
try:
    TZ_NAME
except NameError:
    TZ_NAME = "Europe/Zurich"

try:
    TZ
except NameError:
    TZ = None

if TZ is None:
    try:
        import pytz
        TZ = pytz.timezone(TZ_NAME)
    except Exception:
        from zoneinfo import ZoneInfo
        TZ = ZoneInfo(TZ_NAME)

print("✅ Using timezone:", TZ_NAME)


def _align_datetime_series_to_zurich(s: pd.Series) -> pd.Series:
    """
    Make a datetime series consistently Europe/Zurich timezone-aware.

    - If naive: treat it as Zurich local time
    - If timezone-aware: convert it to Zurich time
    """
    s2 = pd.to_datetime(s, errors="coerce")

    if TZ is None:
        return s2

    try:
        if s2.dt.tz is None:
            return s2.dt.tz_localize(
                TZ_NAME,
                ambiguous="infer",
                nonexistent="shift_forward"
            )
        else:
            return s2.dt.tz_convert(TZ_NAME)
    except Exception:
        # Fallback row-by-row for messy/mixed timezone data
        def fix_one(x):
            if pd.isna(x):
                return pd.NaT
            x = pd.Timestamp(x)
            try:
                if x.tzinfo is None:
                    return x.tz_localize(
                        TZ_NAME,
                        ambiguous="infer",
                        nonexistent="shift_forward"
                    )
                return x.tz_convert(TZ_NAME)
            except Exception:
                return x

        return s.apply(fix_one)


# -------------------------------------------------------------------
# Normalize RDA and Webfleet timestamps before building rda_daily
# This avoids mixing naive, UTC, and Zurich datetimes later in Cell 7.
# -------------------------------------------------------------------
for _df_name, _df in [("rda", rda), ("wf", wf)]:
    for _col in ["start", "end"]:
        if _col in _df.columns:
            _df[_col] = _align_datetime_series_to_zurich(_df[_col])

# Recompute Webfleet date after timezone normalization
if "wf" in globals() and "start" in wf.columns:
    wf["date"] = wf["start"].dt.date
    wf["duration_min"] = (
        pd.to_datetime(wf["end"]) - pd.to_datetime(wf["start"])
    ).dt.total_seconds() / 60.0

    if "km" in wf.columns:
        wf["speed_kmh"] = series_speed_kmh(wf["km"], wf["start"], wf["end"])


def merge_blocks(df, gap_min):
    df = df[["start", "end"]].dropna().sort_values("start")
    if df.empty:
        return []

    merged = []
    cur_s, cur_e = df.iloc[0]["start"], df.iloc[0]["end"]

    for _, r in df.iloc[1:].iterrows():
        s, e = r["start"], r["end"]
        gap = (s - cur_e).total_seconds() / 60.0

        if gap <= gap_min:
            cur_e = max(cur_e, e)
        else:
            merged.append((cur_s, cur_e))
            cur_s, cur_e = s, e

    merged.append((cur_s, cur_e))
    return merged


rows = []

for (cid, day), grp in rda.dropna(
    subset=["collab_id", "jour", "start", "end"]
).groupby(["collab_id", "jour"]):

    grp = grp.copy().sort_values(["start", "end"])

    starts = grp["start"].dropna()
    ends = grp["end"].dropna()

    if starts.empty or ends.empty:
        continue

    first_start = starts.min()
    last_end = ends.max()

    total_min = pd.to_numeric(grp["duree_min"], errors="coerce")
    fallback = (grp["end"] - grp["start"]).dt.total_seconds() / 60.0
    total_min = total_min.fillna(fallback).sum()

    blocks_tight = merge_blocks(grp, gap_min=GAP_MERGE_MIN)
    block_cnt = len(blocks_tight)

    duty_span_min = (
        pd.to_datetime(last_end) - pd.to_datetime(first_start)
    ).total_seconds() / 60.0

    def time_full():
        return total_min >= FULL_DAY_MINUTES

    def blocks_full():
        return block_cnt >= BLOCKS_FULL

    def span_full():
        return (
            block_cnt >= MIN_BLOCKS_FOR_SPAN
            and duty_span_min >= FULL_SPAN_MINUTES
        )

    method = (DAY_CLASS_METHOD or "time").lower()

    if method == "time":
        day_class = "Full" if time_full() else "Half"
    elif method == "blocks":
        day_class = "Full" if blocks_full() else "Half"
    elif method == "span":
        day_class = "Full" if span_full() else "Half"
    else:
        day_class = "Full" if (time_full() or span_full()) else "Half"

    # ---------------------------------------------------------------
    # Internal buffer B
    # First big gap between main work block #1 and main work block #2
    # ---------------------------------------------------------------
    main_blocks = merge_blocks(grp, gap_min=INTERNAL_BLOCK_GAP_MIN)

    raw_ibs = pd.NaT
    raw_ibe = pd.NaT
    ibs = pd.NaT
    ibe = pd.NaT

    if len(main_blocks) >= 2:
        raw_ibs = main_blocks[0][1]
        raw_ibe = main_blocks[1][0]

        ibs = raw_ibs + pd.Timedelta(minutes=INTERNAL_BUFFER_MIN)
        ibe = raw_ibe - pd.Timedelta(minutes=INTERNAL_BUFFER_MIN)

        if pd.notna(ibs) and pd.notna(ibe) and ibe <= ibs:
            raw_ibs = pd.NaT
            raw_ibe = pd.NaT
            ibs = pd.NaT
            ibe = pd.NaT

    rows.append({
        "collab_id": str(cid),
        "date": day,

        "rda_first_start": first_start,
        "rda_last_end": last_end,

        "rda_total_min": float(total_min),
        "rda_block_count": int(block_cnt),
        "duty_span_min": float(duty_span_min),
        "day_class": day_class,

        "buffer_end": last_end + pd.Timedelta(minutes=WORK_END_BUFFER_MIN),

        "buffer_start": (
            first_start - pd.Timedelta(minutes=PRE_SHIFT_BUFFER_MIN)
            if CHECK_PRE_SHIFT
            else pd.NaT
        ),

        "raw_internal_buf_start": raw_ibs,
        "raw_internal_buf_end": raw_ibe,
        "internal_buf_start": ibs,
        "internal_buf_end": ibe,
    })


rda_daily = pd.DataFrame(rows)

# -------------------------------------------------------------------
# Final Zurich alignment for all rda_daily datetime columns
# Do NOT localize to UTC here.
# -------------------------------------------------------------------
datetime_cols = [
    "rda_first_start",
    "rda_last_end",
    "buffer_start",
    "buffer_end",
    "raw_internal_buf_start",
    "raw_internal_buf_end",
    "internal_buf_start",
    "internal_buf_end",
]

for col in datetime_cols:
    if col in rda_daily.columns:
        rda_daily[col] = _align_datetime_series_to_zurich(rda_daily[col])

print("✅ rda_daily rows:", len(rda_daily))
print("✅ rda_daily timezone aligned to:", TZ_NAME)
print("✅ Date range:", rda_daily["date"].min(), "→", rda_daily["date"].max())

✅ Using timezone: Europe/Zurich
✅ rda_daily rows: 1018
✅ rda_daily timezone aligned to: Europe/Zurich
✅ Date range: 2026-04-01 → 2026-04-30


In [ ]:
# @title
# =========================
# Cell 7 — Trip annotation + flags (same set as old)
# =========================
wf = wf.copy()
wf["collab_id"] = wf["collab_id"].astype(str)

merge_cols = [
    "rda_first_start","rda_last_end",
    "buffer_start","buffer_end",
    "internal_buf_start","internal_buf_end",
    "day_class",
    "raw_internal_buf_start","raw_internal_buf_end"
]

# Drop existing merged columns if the cell is run multiple times
wf = wf.drop(columns=[c for c in merge_cols if c in wf.columns], errors="ignore")

wf = wf.merge(
    rda_daily[["collab_id", "date"] + merge_cols],
    how="left",
    on=["collab_id","date"]
)

wf["offday"] = wf["rda_first_start"].isna()
wf["within_service"] = (~wf["offday"]) & (wf["start"] <= wf["rda_last_end"]) & (wf["end"] >= wf["rda_first_start"])
wf["after_buffer"] = (~wf["buffer_end"].isna()) & (wf["start"] >= wf["buffer_end"])

wf["before_shift"] = False
if CHECK_PRE_SHIFT:
    wf["before_shift"] = (~wf["buffer_start"].isna()) & (wf["end"] <= wf["buffer_start"])

def interval_overlap(a_s, a_e, b_s, b_e) -> bool:
    if pd.isna(a_s) or pd.isna(a_e) or pd.isna(b_s) or pd.isna(b_e):
        return False
    overlap_start = max(a_s, b_s)
    overlap_end = min(a_e, b_e)
    overlap_min = (overlap_end - overlap_start).total_seconds()/60.0
    if overlap_min <= 0:
        return False
    if INTERNAL_BUF_MIN_OVERLAP_MIN > 0 and overlap_min < INTERNAL_BUF_MIN_OVERLAP_MIN:
        return False
    if INTERNAL_BUF_MIN_OVERLAP_RATIO > 0:
        trip_min = (a_e - a_s).total_seconds()/60.0
        if trip_min > 0 and (overlap_min / trip_min) < INTERNAL_BUF_MIN_OVERLAP_RATIO:
            return False
    return True

wf["in_internal_buffer"] = [
    interval_overlap(s, e, ibs, ibe)
    for s, e, ibs, ibe in zip(wf["start"], wf["end"], wf["internal_buf_start"], wf["internal_buf_end"])
]

def build_flags(row):
    km = float(row["km"]) if pd.notna(row["km"]) else 0.0
    f = []

    if row["within_service"] and row["tripmode"] == 1 and km >= KM_MIN_FOR_FLAG_CONTEXT:
        f.append("MODE1_DURING_SERVICE")

    if row["after_buffer"] and row["tripmode"] in [2,3] and km >= KM_MIN_FOR_FLAG_CONTEXT:
        f.append("MODE23_AFTER_BUFFER")

    if row["offday"] and row["tripmode"] in [2,3] and km >= KM_MIN_FOR_FLAG_CONTEXT:
        f.append("MODE23_ON_OFFDAY")
        f.append("NO_RDA_BUT_MODE23")

    if row["before_shift"] and row["tripmode"] in [2,3] and km >= KM_MIN_FOR_FLAG_CONTEXT:
        f.append("PRE_SHIFT_MODE2")

    if row["in_internal_buffer"] and row["tripmode"] in [2,3] and km >= KM_MIN_FOR_FLAG_CONTEXT:
        f.append("MODE23_IN_INTERNAL_BUFFER")

    if pd.isna(row["collab_id"]) or row["collab_id"] == "None" or row["collab_id"] == "nan":
        f.append("UNMAPPED_DRIVER")

    if pd.notna(row.get("speed_kmh", np.nan)) and row["speed_kmh"] > MAX_REASONABLE_SPEED_KMH:
        f.append("SPEED_EXCEEDS_MAX")

    # Extra: on RDA day, mode2/3 outside service (after buffer OR before shift)
    has_rda_today = not bool(row["offday"])
    outside_service = (not bool(row["within_service"])) and (bool(row["after_buffer"]) or bool(row["before_shift"]))
    if has_rda_today and outside_service and row["tripmode"] in [2,3] and km >= KM_MIN_FOR_FLAG_CONTEXT:
        f.append("MODE23_OUTSIDE_SERVICE_ON_RDA_DAY")

    return ",".join(sorted(set(f)))

wf["flags"] = wf.apply(build_flags, axis=1)

wf["suspect_private_km"] = np.where(
    wf["tripmode"].isin([2,3]) & (wf["after_buffer"] | wf["offday"] | wf["before_shift"] | wf["in_internal_buffer"]),
    wf["km"].fillna(0.0),
    0.0
)

print("✅ Trip flags built. Top flags:", wf["flags"].value_counts().head(10).to_dict())

✅ Trip flags built. Top flags: {'': 5089, 'MODE23_ON_OFFDAY,NO_RDA_BUT_MODE23,UNMAPPED_DRIVER': 328, 'MODE23_ON_OFFDAY,NO_RDA_BUT_MODE23': 233, 'MODE23_AFTER_BUFFER,MODE23_OUTSIDE_SERVICE_ON_RDA_DAY': 165, 'UNMAPPED_DRIVER': 82, 'MODE23_IN_INTERNAL_BUFFER': 71, 'MODE1_DURING_SERVICE': 55, 'MODE23_OUTSIDE_SERVICE_ON_RDA_DAY,PRE_SHIFT_MODE2': 45}


In [ ]:
# @title
# =========================
# Cell 7b — Daily usage labels (same set)
# =========================
rda_dates = rda[["collab_id","jour"]].dropna().copy()
rda_dates["collab_id"] = rda_dates["collab_id"].astype(str)
rda_dates["date"] = rda_dates["jour"]
rda_dates = rda_dates[["collab_id","date"]]

wf_dates = wf[["collab_id"]].copy()
wf_dates["collab_id"] = wf_dates["collab_id"].astype(str)
wf_dates["date"] = wf["start"].dt.date

all_days = pd.concat([rda_dates, wf_dates], ignore_index=True).dropna(subset=["collab_id","date"]).drop_duplicates()
all_days["date"] = pd.to_datetime(all_days["date"]).dt.date

wf_day_agg = wf.groupby(["collab_id","date"]).agg(
    wf_trip_count=("tripid","count"),
    wf_km_total=("km","sum"),
    wf_any_within_service=("within_service", lambda s: bool(pd.Series(s).fillna(False).any())),
    wf_any_after_buffer=("after_buffer", lambda s: bool(pd.Series(s).fillna(False).any())),
    wf_any_before_shift=("before_shift", lambda s: bool(pd.Series(s).fillna(False).any())),
).reset_index()

ad = (all_days
      .merge(rda_daily[["collab_id","date","rda_first_start","rda_last_end","day_class"]], how="left", on=["collab_id","date"])
      .merge(wf_day_agg, how="left", on=["collab_id","date"])
)

ad["wf_trip_count"] = ad["wf_trip_count"].fillna(0).astype(int)
ad["wf_km_total"] = ad["wf_km_total"].fillna(0.0).astype(float)
for c in ["wf_any_within_service","wf_any_after_buffer","wf_any_before_shift"]:
    ad[c] = ad[c].fillna(False).astype(bool)

ad["has_rda"] = ad["rda_first_start"].notna()
ad["has_wf"] = ad["wf_trip_count"] > 0
ad["car_used_within_service"] = ad["has_rda"] & ad["wf_any_within_service"]
ad["car_used_outside_service"] = ad["has_rda"] & (ad["wf_any_after_buffer"] | ad["wf_any_before_shift"])

def usage_label(row):
    if row["has_rda"]:
        if not row["has_wf"]:
            return "WORK_NO_CAR"
        if row["car_used_within_service"]:
            return "WORK_CAR_IN_SERVICE"
        return "WORK_CAR_ONLY_OUTSIDE"
    else:
        return "NO_WORK_CAR_USED" if row["has_wf"] else "OFF_NO_CAR"

ad["usage_label"] = ad.apply(usage_label, axis=1)
daily_usage = ad[["collab_id","date","has_rda","has_wf","car_used_within_service","car_used_outside_service","usage_label","day_class"]].copy()

print("✅ daily_usage ready. Sample:")
print(daily_usage.head(10))

✅ daily_usage ready. Sample:
     collab_id        date  has_rda  has_wf  car_used_within_service  car_used_outside_service          usage_label day_class
0  collab-0194  2026-04-01     True   False                    False                     False          WORK_NO_CAR      Full
1  collab-0243  2026-04-01     True   False                    False                     False          WORK_NO_CAR      Half
2  collab-0213  2026-04-01     True   False                    False                     False          WORK_NO_CAR      Full
3  collab-0255  2026-04-01     True    True                     True                     False  WORK_CAR_IN_SERVICE      Full
4  collab-0187  2026-04-01     True   False                    False                     False          WORK_NO_CAR      Half
5  collab-0200  2026-04-01     True    True                     True                      True  WORK_CAR_IN_SERVICE      Half
6  collab-0196  2026-04-06     True    True                     True                     

/tmp/ipykernel_25424/1130323736.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ad[c] = ad[c].fillna(False).astype(bool)
/tmp/ipykernel_25424/1130323736.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ad[c] = ad[c].fillna(False).astype(bool)
/tmp/ipykernel_25424/1130323736.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ad[

In [ ]:
# @title
# =========================
# Cell 7c — 61010 strict check + deviation metrics for PDF labels — TZ safe
# =========================

prestation61010_checks = pd.DataFrame()
prestation61010_day_counts = pd.DataFrame()

# ------------------------------------------------------------
# Helper: compare everything in this cell as local Zurich time
# without timezone attached.
#
# This avoids errors like:
# "Cannot compare tz-naive and tz-aware timestamps"
# ------------------------------------------------------------
try:
    TZ_NAME
except NameError:
    TZ_NAME = "Europe/Zurich"

def _one_to_local_naive(x):
    if pd.isna(x):
        return pd.NaT

    ts = pd.Timestamp(x)

    try:
        if ts.tzinfo is not None:
            return ts.tz_convert(TZ_NAME).tz_localize(None)
    except Exception:
        try:
            return ts.tz_localize(None)
        except Exception:
            pass

    return ts

def _series_to_local_naive(s):
    return pd.to_datetime(s.apply(_one_to_local_naive), errors="coerce")

def gap_min(tr_s, tr_e, win_s, win_e):
    if pd.isna(tr_s) or pd.isna(tr_e) or pd.isna(win_s) or pd.isna(win_e):
        return np.nan

    if tr_e < win_s:
        return (win_s - tr_e).total_seconds() / 60.0

    if tr_s > win_e:
        return (tr_s - win_e).total_seconds() / 60.0

    return 0.0

def overlap_min(a_s, a_e, b_s, b_e):
    if pd.isna(a_s) or pd.isna(a_e) or pd.isna(b_s) or pd.isna(b_e):
        return np.nan

    ov_s = max(a_s, b_s)
    ov_e = min(a_e, b_e)

    return max(0.0, (ov_e - ov_s).total_seconds() / 60.0)


if not ENABLE_61010_FEATURE:
    print("61010 disabled.")

else:
    # ------------------------------------------------------------
    # Prepare RDA 61010 rows
    # ------------------------------------------------------------
    r61010 = rda[
        (rda["prestation_code"].astype(str) == str(PRESTATION_61010_CODE)) &
        rda["collab_id"].notna() &
        rda["start"].notna() &
        rda["end"].notna()
    ].copy()

    r61010["collab_id"] = r61010["collab_id"].astype(str)
    r61010["start"] = _series_to_local_naive(r61010["start"])
    r61010["end"] = _series_to_local_naive(r61010["end"])

    # Safety: remove invalid 61010 rows
    r61010 = r61010[
        r61010["start"].notna() &
        r61010["end"].notna() &
        (r61010["end"] > r61010["start"])
    ].copy()

    print("✅ 61010 prestations found:", len(r61010))

    # ------------------------------------------------------------
    # Prepare Webfleet rows
    # ------------------------------------------------------------
    wf_ok = wf[
        wf["collab_id"].notna() &
        wf["start"].notna() &
        wf["end"].notna()
    ].copy()

    wf_ok["collab_id"] = wf_ok["collab_id"].astype(str)
    wf_ok["start"] = _series_to_local_naive(wf_ok["start"])
    wf_ok["end"] = _series_to_local_naive(wf_ok["end"])

    wf_ok = wf_ok[
        wf_ok["start"].notna() &
        wf_ok["end"].notna() &
        (wf_ok["end"] > wf_ok["start"])
    ].copy()

    # Index Webfleet trips by collaborator for faster lookup
    wf_groups = {
        cid: g.sort_values("start").copy()
        for cid, g in wf_ok.groupby("collab_id")
    }

    rows = []

    # ------------------------------------------------------------
    # Main 61010 check
    # ------------------------------------------------------------
    for rr in r61010.itertuples(index=False):
        cid = str(rr.collab_id)
        st = rr.start
        en = rr.end

        buf_s = st - pd.Timedelta(minutes=PRESTATION_61010_BUFFER_MIN)
        buf_e = en + pd.Timedelta(minutes=PRESTATION_61010_BUFFER_MIN)

        trips = wf_groups.get(cid, pd.DataFrame())

        has_inside = False
        nearest_gap = np.nan
        nearest_tripid = None
        nearest_tripmode = None
        nearest_trip_start = pd.NaT
        nearest_trip_end = pd.NaT

        nearest_overlap_buf_min = np.nan
        nearest_overlap_61010_min = np.nan

        start_delta_rda_vs_wf_min = np.nan
        end_delta_rda_vs_wf_min = np.nan

        rda_61010_duration_min = max(
            0.0,
            (en - st).total_seconds() / 60.0
        )

        uncovered_61010_min = np.nan
        total_deviation_min = np.nan

        if not trips.empty:
            # Candidate trips within a wide search window around this 61010
            cand = trips[
                (trips["start"] <= buf_e + pd.Timedelta(hours=12)) &
                (trips["end"] >= buf_s - pd.Timedelta(hours=12))
            ].copy()

            if not cand.empty:
                # Strict check:
                # A WF trip must be fully inside the 61010 ± buffer window.
                inside = (
                    (cand["start"] >= buf_s) &
                    (cand["end"] <= buf_e)
                )

                nearest_row = None

                if inside.any():
                    has_inside = True
                    nearest_gap = 0.0
                    nearest_row = cand.loc[inside].sort_values("start").iloc[0]

                else:
                    cand["gap_min"] = [
                        gap_min(ts, te, buf_s, buf_e)
                        for ts, te in zip(cand["start"], cand["end"])
                    ]

                    cand = cand.dropna(subset=["gap_min"]).sort_values(
                        ["gap_min", "start"]
                    )

                    if not cand.empty:
                        nearest_row = cand.iloc[0]
                        nearest_gap = float(nearest_row.get("gap_min", np.nan))

                if nearest_row is not None:
                    nearest_tripid = str(nearest_row.get("tripid", ""))
                    nearest_tripmode = nearest_row.get("tripmode", None)
                    nearest_trip_start = nearest_row.get("start", pd.NaT)
                    nearest_trip_end = nearest_row.get("end", pd.NaT)

                    nearest_overlap_buf_min = overlap_min(
                        nearest_trip_start,
                        nearest_trip_end,
                        buf_s,
                        buf_e
                    )

                    nearest_overlap_61010_min = overlap_min(
                        nearest_trip_start,
                        nearest_trip_end,
                        st,
                        en
                    )

                    if pd.notna(nearest_trip_start):
                        start_delta_rda_vs_wf_min = (
                            st - nearest_trip_start
                        ).total_seconds() / 60.0

                    if pd.notna(nearest_trip_end):
                        end_delta_rda_vs_wf_min = (
                            en - nearest_trip_end
                        ).total_seconds() / 60.0

                    # Metric 1:
                    # How many minutes of the actual 61010 block are not covered
                    # by the nearest Webfleet trip?
                    nearest_overlap_for_calc = (
                        nearest_overlap_61010_min
                        if pd.notna(nearest_overlap_61010_min)
                        else 0.0
                    )

                    uncovered_61010_min = max(
                        0.0,
                        rda_61010_duration_min - nearest_overlap_for_calc
                    )

                    # Metric 2:
                    # Start/end drift between the 61010 row and the nearest trip
                    if (
                        pd.notna(start_delta_rda_vs_wf_min)
                        and pd.notna(end_delta_rda_vs_wf_min)
                    ):
                        total_deviation_min = (
                            abs(start_delta_rda_vs_wf_min)
                            + abs(end_delta_rda_vs_wf_min)
                        )

        # Fallbacks when no nearest trip was found
        if pd.isna(uncovered_61010_min):
            uncovered_61010_min = rda_61010_duration_min

        if pd.isna(total_deviation_min):
            total_deviation_min = rda_61010_duration_min

        rows.append({
            "rda_row_id": int(rr.rda_row_id),
            "collab_id": cid,
            "date": pd.to_datetime(st).date(),

            "start": st,
            "end": en,

            "buf_start": buf_s,
            "buf_end": buf_e,

            "has_wf_trip_in_buffer": bool(has_inside),
            "nearest_gap_min": nearest_gap,

            "nearest_tripid": nearest_tripid,
            "nearest_tripmode": nearest_tripmode,
            "nearest_trip_start": nearest_trip_start,
            "nearest_trip_end": nearest_trip_end,

            "rda_61010_duration_min": rda_61010_duration_min,
            "nearest_overlap_buf_min": nearest_overlap_buf_min,
            "nearest_overlap_61010_min": nearest_overlap_61010_min,

            "start_delta_rda_vs_wf_min": start_delta_rda_vs_wf_min,
            "end_delta_rda_vs_wf_min": end_delta_rda_vs_wf_min,

            "uncovered_61010_min": uncovered_61010_min,
            "total_deviation_min": total_deviation_min,
        })

    prestation61010_checks = pd.DataFrame(rows)

    # ------------------------------------------------------------
    # Daily counts
    # ------------------------------------------------------------
    if not prestation61010_checks.empty:
        prestation61010_day_counts = (
            prestation61010_checks
            .groupby(["collab_id", "date"])
            .agg(
                prestation_61010_count=("rda_row_id", "count"),
                prestation_61010_missing=(
                    "has_wf_trip_in_buffer",
                    lambda s: int((~pd.Series(s).astype(bool)).sum())
                ),
            )
            .reset_index()
        )

    # ------------------------------------------------------------
    # Merge 61010 check columns back into rda for later plotting/export.
    # Drop previous versions first so rerunning the cell does not create
    # _x / _y duplicate columns.
    # ------------------------------------------------------------
    merge_back_cols = [
        "has_wf_trip_in_buffer",
        "buf_start",
        "buf_end",
        "rda_61010_duration_min",
        "nearest_overlap_buf_min",
        "nearest_overlap_61010_min",
        "start_delta_rda_vs_wf_min",
        "end_delta_rda_vs_wf_min",
        "uncovered_61010_min",
        "total_deviation_min",
    ]

    rda = rda.drop(
        columns=[c for c in merge_back_cols if c in rda.columns],
        errors="ignore"
    )

    if not prestation61010_checks.empty:
        rda = rda.merge(
            prestation61010_checks[
                ["rda_row_id"] + merge_back_cols
            ],
            how="left",
            on="rda_row_id"
        )
    else:
        for c in merge_back_cols:
            rda[c] = np.nan

    missing_61010_count = 0
    if not prestation61010_checks.empty:
        missing_61010_count = int(
            (~prestation61010_checks["has_wf_trip_in_buffer"].astype(bool)).sum()
        )

    print("❗61010 missing:", missing_61010_count)
    print("✅ prestation61010_checks rows:", len(prestation61010_checks))
    print("✅ prestation61010_day_counts rows:", len(prestation61010_day_counts))
    print("✅ 7c done — timezone-safe.")

✅ 61010 prestations found: 3614
❗61010 missing: 1535
✅ prestation61010_checks rows: 3614
✅ prestation61010_day_counts rows: 556
✅ 7c done — timezone-safe.


In [ ]:
# @title
# =========================
# Cell 8 — agg_daily + collab_stats (stable)
# =========================
wf2 = wf.copy()
wf2["collab_id"] = wf2["collab_id"].astype(str)
wf2["km"] = pd.to_numeric(wf2["km"], errors="coerce").fillna(0.0)

wf2["km_m1"] = np.where(wf2["tripmode"] == 1, wf2["km"], 0.0)
wf2["km_m2"] = np.where(wf2["tripmode"] == 2, wf2["km"], 0.0)
wf2["km_m3"] = np.where(wf2["tripmode"] == 3, wf2["km"], 0.0)

wf2["trips_m1"] = (wf2["tripmode"] == 1).astype(int)
wf2["trips_m2"] = (wf2["tripmode"] == 2).astype(int)
wf2["trips_m3"] = (wf2["tripmode"] == 3).astype(int)

agg_daily = wf2.groupby(["collab_id","date"], dropna=False).agg(
    wf_trip_count=("tripid","count"),
    trips_m1=("trips_m1","sum"),
    trips_m2=("trips_m2","sum"),
    trips_m3=("trips_m3","sum"),
    km_mode1=("km_m1","sum"),
    km_mode2=("km_m2","sum"),
    km_mode3=("km_m3","sum"),
    suspect_private_km=("suspect_private_km","sum"),
    flags_concat=("flags", uniq_flags),
).reset_index()

agg_daily = agg_daily.merge(rda_daily, how="left", on=["collab_id","date"])
agg_daily["private_km_total_for_day"] = (agg_daily["km_mode1"].fillna(0) + agg_daily["suspect_private_km"].fillna(0)).round(3)

# add daily usage label
agg_daily = agg_daily.merge(daily_usage[["collab_id","date","usage_label","has_rda","has_wf","car_used_within_service","car_used_outside_service"]], how="left", on=["collab_id","date"])

# names
name_cols = map_df[["collab_id","collab_name_sarl","collab_no_sarl","collab_name_wf","driverno"]].drop_duplicates()
name_cols["collab_id"] = name_cols["collab_id"].astype(str)
agg_daily = agg_daily.merge(name_cols, how="left", on="collab_id")

# collab_stats
collab_stats = wf2.groupby("collab_id", dropna=False).agg(
    trips_total=("tripid","count"),
    trips_m1=("trips_m1","sum"),
    trips_m2=("trips_m2","sum"),
    trips_m3=("trips_m3","sum"),
    km_total=("km","sum"),
    km_m1=("km_m1","sum"),
    km_m2=("km_m2","sum"),
    km_m3=("km_m3","sum"),
    suspect_private_km=("suspect_private_km","sum"),
    flags_concat=("flags", uniq_flags),
).reset_index().merge(name_cols, how="left", on="collab_id")

# day counts full/half
if not rda_daily.empty:
    day_counts = rda_daily.pivot_table(index="collab_id", columns="day_class", values="date", aggfunc="count").fillna(0).astype(int)
    day_counts = day_counts.rename(columns={"Full":"days_full","Half":"days_half"})
    for c in ["days_full","days_half"]:
        if c not in day_counts.columns:
            day_counts[c] = 0
    collab_stats = collab_stats.merge(day_counts.reset_index(), how="left", on="collab_id")
else:
    collab_stats["days_full"] = 0
    collab_stats["days_half"] = 0

# usage counts per collaborator
usage_counts = agg_daily.groupby("collab_id")["usage_label"].value_counts().unstack(fill_value=0)
for col in ["WORK_NO_CAR","NO_WORK_CAR_USED","WORK_CAR_IN_SERVICE","WORK_CAR_ONLY_OUTSIDE","OFF_NO_CAR"]:
    if col not in usage_counts.columns:
        usage_counts[col] = 0
usage_counts = usage_counts.reset_index()
collab_stats = collab_stats.merge(usage_counts, how="left", on="collab_id")
for col in ["WORK_NO_CAR","NO_WORK_CAR_USED","WORK_CAR_IN_SERVICE","WORK_CAR_ONLY_OUTSIDE","OFF_NO_CAR"]:
    collab_stats[col] = collab_stats[col].fillna(0).astype(int)

# work days with car by Full/Half
base = agg_daily.copy()
base["car_used_within_service"] = base["car_used_within_service"].fillna(False).astype(bool)
full_days_with_car = base[(base["day_class"]=="Full") & (base["car_used_within_service"])].groupby("collab_id")["date"].nunique()
half_days_with_car = base[(base["day_class"]=="Half") & (base["car_used_within_service"])].groupby("collab_id")["date"].nunique()
counts_car = pd.DataFrame({
    "collab_id": collab_stats["collab_id"],
    "full_days_with_car": collab_stats["collab_id"].map(full_days_with_car).fillna(0).astype(int),
    "half_days_with_car": collab_stats["collab_id"].map(half_days_with_car).fillna(0).astype(int),
})
collab_stats = collab_stats.merge(counts_car, how="left", on="collab_id")
collab_stats["work_days_total"] = (collab_stats["days_full"].fillna(0).astype(int) + collab_stats["days_half"].fillna(0).astype(int))

# 61010 totals
if ENABLE_61010_FEATURE and not prestation61010_checks.empty:
    tot = prestation61010_checks.groupby("collab_id").agg(
        prestation_61010_total=("rda_row_id","count"),
        prestation_61010_missing_total=("has_wf_trip_in_buffer", lambda s: int((~pd.Series(s).astype(bool)).sum()))
    ).reset_index()
    collab_stats = collab_stats.merge(tot, how="left", on="collab_id")
else:
    collab_stats["prestation_61010_total"] = 0
    collab_stats["prestation_61010_missing_total"] = 0

collab_stats["prestation_61010_total"] = collab_stats.get("prestation_61010_total", 0).fillna(0).astype(int)
collab_stats["prestation_61010_missing_total"] = collab_stats.get("prestation_61010_missing_total", 0).fillna(0).astype(int)

print("✅ Aggregations ready:", len(agg_daily), "daily rows |", len(collab_stats), "collabs")

✅ Aggregations ready: 666 daily rows | 43 collabs


In [ ]:
# @title
# =========================
# Cell 9 — Search helpers
# =========================
def search_trips(df_trips, collab_ids=None, date_start=None, date_end=None, modes=None, min_km=0.0, has_flags=None):
    q = df_trips.copy()
    if collab_ids is not None:
        q = q[q["collab_id"].isin(collab_ids)]
    if date_start is not None:
        ds = pd.to_datetime(date_start).date()
        q = q[q["start"].dt.date >= ds]
    if date_end is not None:
        de = pd.to_datetime(date_end).date()
        q = q[q["start"].dt.date <= de]
    if modes is not None:
        q = q[q["tripmode"].isin(modes)]
    if min_km and min_km > 0:
        q = q[q["km"] >= min_km]
    if has_flags:
        def ok(s):
            if pd.isna(s) or not str(s).strip():
                return False
            parts = str(s).split(",")
            return any(f in parts for f in has_flags)
        q = q[q["flags"].apply(ok)]
    return q.sort_values(["collab_id","start"])

def search_days(df_days, collab_ids=None, date_start=None, date_end=None, day_class=None, min_private_km=None, has_flags=None):
    q = df_days.copy()
    if collab_ids is not None:
        q = q[q["collab_id"].isin(collab_ids)]
    if date_start is not None:
        ds = pd.to_datetime(date_start).date()
        q = q[q["date"] >= ds]
    if date_end is not None:
        de = pd.to_datetime(date_end).date()
        q = q[q["date"] <= de]
    if day_class is not None:
        if isinstance(day_class, str):
            q = q[q["day_class"] == day_class]
        else:
            q = q[q["day_class"].isin(day_class)]
    if min_private_km is not None:
        q = q[q["private_km_total_for_day"] >= float(min_private_km)]
    if has_flags:
        def ok(s):
            if pd.isna(s) or not str(s).strip():
                return False
            parts = str(s).split(",")
            return any(f in parts for f in has_flags)
        q = q[q["flags_concat"].apply(ok)]
    return q.sort_values(["collab_id","date"])

print("✅ Search helpers ready.")

✅ Search helpers ready.


In [ ]:
# @title
# =========================
# Cell 10 — Excel export (tz-safe) + Flag_Summary + 61010 sheets
# =========================
global_path = os.path.join(GLOBAL_DIR, "webfleet_rda_audit_global_all_in_one.xlsx")

# tz-safe copies
wf_x = drop_tz_excel_safe(wf2)
rda_x = drop_tz_excel_safe(rda)
rda_daily_x = drop_tz_excel_safe(rda_daily)
agg_daily_x = drop_tz_excel_safe(agg_daily)
collab_stats_x = drop_tz_excel_safe(collab_stats)
planning_x = drop_tz_excel_safe(planning)
map_x = map_df.copy()

# Flag summary
flags_exp = wf[["collab_id","flags","km"]].copy()
flags_exp["flag"] = flags_exp["flags"].fillna("").astype(str).str.split(",")
flags_exp = flags_exp.explode("flag")
flags_exp = flags_exp[flags_exp["flag"].astype(str).str.strip() != ""]
flag_summary = (flags_exp.groupby("flag").agg(count=("collab_id","count"), km_sum=("km","sum")).reset_index()
                .sort_values(["count","km_sum"], ascending=[False,False]))

with pd.ExcelWriter(global_path, engine="openpyxl") as xw:
    collab_stats_x.to_excel(xw, index=False, sheet_name="Collaborator_Summary")
    agg_daily_x.to_excel(xw, index=False, sheet_name="Daily_Aggregates")
    wf_x.to_excel(xw, index=False, sheet_name="All_Trips")
    rda_x.to_excel(xw, index=False, sheet_name="All_RDA_Entries")
    rda_daily_x.to_excel(xw, index=False, sheet_name="RDA_Daily")
    map_x.to_excel(xw, index=False, sheet_name="Mapping")
    flag_summary.to_excel(xw, index=False, sheet_name="Flag_Summary")
    planning_x.to_excel(xw, index=False, sheet_name="Planning")

    if ENABLE_61010_FEATURE and not prestation61010_checks.empty:
        drop_tz_excel_safe(prestation61010_checks).to_excel(xw, index=False, sheet_name="Prestation_61010_Checks")
    if ENABLE_61010_FEATURE and not prestation61010_day_counts.empty:
        prestation61010_day_counts.to_excel(xw, index=False, sheet_name="Prestation_61010_DayCounts")

print("✅ Excel written:", global_path)

/tmp/ipykernel_25424/3385414843.py:146: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if c in out.columns and pdt.is_datetime64tz_dtype(out[c]):
/tmp/ipykernel_25424/3385414843.py:146: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if c in out.columns and pdt.is_datetime64tz_dtype(out[c]):


✅ Excel written: /content/drive/MyDrive/sarl year 2025/Webfleet_RDA_Audit_20260512_101940/02_Global/webfleet_rda_audit_global_all_in_one.xlsx


In [ ]:
# =========================
# New Cell A — Modified/Adjusted RDA DISABLED
# =========================
print("✅ Modified/Adjusted RDA is disabled.")
print("✅ No adjusted RDA file will be created.")
print("✅ No modified RDA data will be used anywhere in the graphics.")

✅ Modified/Adjusted RDA is disabled.
✅ No adjusted RDA file will be created.
✅ No modified RDA data will be used anywhere in the graphics.


In [ ]:
# =========================
# New Cell B — Comparison PDF export with 3 lanes
# WF / Original RDA / Planning
# Modified RDA completely removed
# =========================

from pathlib import Path
import re
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.backends.backend_pdf import PdfPages

COMPARE_PDF_DIR = Path(OUTPUT_DIR) / "05_Plots_PDF_Comparison"
COMPARE_PDF_DIR.mkdir(parents=True, exist_ok=True)

FIG_W = 19.0
FIG_H = 9.4

LANE_Y_CMP = {
    "WF": 0.0,
    "RDA_ORIG": 1.0,
    "Planning": 2.0,
}

LANE_LABELS_CMP = {
    0.0: "WF Trips",
    1.0: "RDA Original",
    2.0: "Planning",
}


# ============================================================
# Safety fallbacks for helpers already defined earlier
# ============================================================

if "_safe_text" not in globals():
    def _safe_text(x):
        if pd.isna(x):
            return ""
        return str(x)

if "_safe_filename" not in globals():
    def _safe_filename(x):
        s = _safe_text(x).strip()
        s = re.sub(r"[^\w\-. ]+", "_", s)
        s = re.sub(r"\s+", "_", s)
        return s[:180] if s else "collaborateur"

if "_to_local_naive" not in globals():
    def _to_local_naive(ts):
        if pd.isna(ts):
            return pd.NaT
        ts = pd.Timestamp(ts)
        try:
            if ts.tzinfo is not None:
                if "TZ" in globals() and TZ is not None:
                    return ts.tz_convert(TZ).tz_localize(None)
                return ts.tz_localize(None)
        except Exception:
            try:
                return ts.tz_localize(None)
            except Exception:
                pass
        return ts

if "_duration_mins" not in globals():
    def _duration_mins(a, b):
        if pd.isna(a) or pd.isna(b):
            return np.nan
        return (pd.Timestamp(b) - pd.Timestamp(a)).total_seconds() / 60.0

if "_fmt_hhmm" not in globals():
    def _fmt_hhmm(ts):
        ts = _to_local_naive(ts)
        if pd.isna(ts):
            return "-"
        return pd.Timestamp(ts).strftime("%H:%M")

if "_fmt_hours" not in globals():
    def _fmt_hours(delta):
        if delta is None or pd.isna(delta):
            return "-"
        total_min = int(round(delta.total_seconds() / 60.0))
        if total_min < 0:
            return "-"
        hh = total_min // 60
        mm = total_min % 60
        return f"{hh}h{mm:02d}" if mm else f"{hh}h"

if "_fmt_span" not in globals():
    def _fmt_span(s, e):
        s = _to_local_naive(s)
        e = _to_local_naive(e)
        if pd.isna(s) or pd.isna(e) or e <= s:
            return "-"
        return f"{_fmt_hhmm(s)} → {_fmt_hhmm(e)} ({_fmt_hours(e - s)})"


# ============================================================
# Robust scalar date helper
# ============================================================

def _cmp_date_str(value, fallback_ts=None):
    """
    Return YYYY-MM-DD from:
      - RDA jour
      - Planning date_only / date
      - fallback timestamp
    """
    if value is not None and not pd.isna(value):
        d = pd.to_datetime(value, errors="coerce", dayfirst=True)
        if pd.notna(d):
            return pd.Timestamp(d).strftime("%Y-%m-%d")

    if fallback_ts is not None and not pd.isna(fallback_ts):
        fb = _to_local_naive(fallback_ts)
        if pd.notna(fb):
            return pd.Timestamp(fb).strftime("%Y-%m-%d")

    return None


# ============================================================
# Collaborator label map
# ============================================================

def _cmp_collab_label_map():
    out = {}
    map_local = map_df.copy() if "map_df" in globals() else pd.DataFrame(columns=["collab_id"])

    if not map_local.empty and "collab_id" in map_local.columns:
        map_local["collab_id"] = map_local["collab_id"].astype(str)

    all_ids = set()

    for df_name in ["wf", "rda", "planning"]:
        if df_name in globals():
            df = globals()[df_name]
            if isinstance(df, pd.DataFrame) and not df.empty and "collab_id" in df.columns:
                all_ids.update(df["collab_id"].dropna().astype(str).unique().tolist())

    for cid in sorted(all_ids):
        label = None

        if not map_local.empty:
            row = map_local[map_local["collab_id"] == cid]

            if not row.empty:
                rr = row.iloc[0]

                nm = _safe_text(rr.get("collab_name_sarl", "")).strip()
                if not nm:
                    nm = _safe_text(rr.get("collab_name_wf", "")).strip()

                sa = _safe_text(rr.get("collab_no_sarl", "")).strip() or "-"
                wf_no = _safe_text(rr.get("driverno", "")).strip() or "-"

                if nm:
                    label = f"{nm} (ID collab-{cid}, SA:{sa}, WF:{wf_no})"

        if not label:
            label = f"Collaborateur {cid}"

        out[cid] = label

    return out


CMP_COLLAB_LABELS = _cmp_collab_label_map()


# ============================================================
# Color helpers
# ============================================================

def _cmp_planning_color(row):
    if str(row.get("client_absent", "")).strip().upper() == "Y":
        return "#000000"

    key = str(row.get("event_color_key", "")).strip().lower()

    plan_color_map_local = PLAN_COLOR_MAP if "PLAN_COLOR_MAP" in globals() else {}

    if key in plan_color_map_local:
        return plan_color_map_local[key]

    return row.get("plot_color", "#bdbdbd")


def _cmp_rda_orig_color(code):
    code = str(code).strip() if pd.notna(code) else ""

    # Special RDA prestation codes shown in yellow
    if code in ["16009", "95900"]:
        return "#FFD700"

    # 61010 stays purple
    if code == str(PRESTATION_61010_CODE):
        return "#800080"

    # Normal RDA stays green
    return "#2ca02c"

# ============================================================
# Monthly KM stats for header
# ============================================================

def _cmp_month_km_stats(collab_id, date_str):
    out = {
        "month_label": "-",
        "km_private": 0.0,
        "km_professional": 0.0,
        "km_commute": 0.0,
        "km_total": 0.0,
        "km_suspect_private": 0.0,
        "km_private_plus_suspect": 0.0,
    }

    if "agg_daily" not in globals() or agg_daily is None or agg_daily.empty:
        return out

    base_day = pd.Timestamp(str(date_str))
    month_period = base_day.to_period("M")
    out["month_label"] = base_day.strftime("%B %Y")

    sub = agg_daily.copy()
    sub["collab_id"] = sub["collab_id"].astype(str)
    sub["date"] = pd.to_datetime(sub["date"], errors="coerce")

    sub = sub[
        (sub["collab_id"] == str(collab_id)) &
        (sub["date"].dt.to_period("M") == month_period)
    ].copy()

    out["km_private"] = float(sub["km_mode1"].fillna(0).sum()) if "km_mode1" in sub.columns else 0.0
    out["km_professional"] = float(sub["km_mode2"].fillna(0).sum()) if "km_mode2" in sub.columns else 0.0
    out["km_commute"] = float(sub["km_mode3"].fillna(0).sum()) if "km_mode3" in sub.columns else 0.0
    out["km_suspect_private"] = float(sub["suspect_private_km"].fillna(0).sum()) if "suspect_private_km" in sub.columns else 0.0
    out["km_private_plus_suspect"] = float(sub["private_km_total_for_day"].fillna(0).sum()) if "private_km_total_for_day" in sub.columns else 0.0

    out["km_total"] = (
        out["km_private"] +
        out["km_professional"] +
        out["km_commute"]
    )

    return out


# ============================================================
# Bounds and span helpers
# ============================================================

def _cmp_day_bounds(day_events, day_str, min_hours=16):
    if day_events.empty:
        base = pd.Timestamp(f"{day_str} 00:00:00")
        return base, base + pd.Timedelta(hours=min_hours)

    left = _to_local_naive(day_events["left"].min())
    right = _to_local_naive(day_events["right"].max())

    if pd.isna(left) or pd.isna(right) or right <= left:
        base = pd.Timestamp(f"{day_str} 00:00:00")
        return base, base + pd.Timedelta(hours=min_hours)

    min_span = pd.Timedelta(hours=min_hours)

    if (right - left) < min_span:
        right = left + min_span

    return left, right


def _cmp_lane_span(day_events, kind):
    sub = day_events[day_events["kind"] == kind].copy()

    if sub.empty:
        return pd.NaT, pd.NaT

    s = _to_local_naive(sub["left"].min())
    e = _to_local_naive(sub["right"].max())

    if pd.isna(s) or pd.isna(e) or e <= s:
        return pd.NaT, pd.NaT

    return s, e


# ============================================================
# RDA rest before/after each day
# ============================================================

def _cmp_fmt_dt_short(ts):
    ts = _to_local_naive(ts)

    if pd.isna(ts):
        return "-"

    return pd.Timestamp(ts).strftime("%d/%m %H:%M")


def _cmp_build_rda_rest_lookup():
    """
    Builds lookup for each collaborator/day:

    previous RDA day last end -> current RDA day first start
    current RDA day last end -> next RDA day first start
    """
    if "rda" not in globals() or rda is None or rda.empty:
        return {}

    rd = rda.copy()

    rd = rd[
        rd["collab_id"].notna() &
        rd["start"].notna() &
        rd["end"].notna()
    ].copy()

    if rd.empty:
        return {}

    rd["collab_id"] = rd["collab_id"].astype(str)
    rd["start"] = rd["start"].apply(_to_local_naive)
    rd["end"] = rd["end"].apply(_to_local_naive)

    rd = rd[
        rd["start"].notna() &
        rd["end"].notna() &
        (rd["end"] > rd["start"])
    ].copy()

    if "jour" in rd.columns:
        rd["date_str"] = rd.apply(
            lambda r: _cmp_date_str(r.get("jour", pd.NaT), fallback_ts=r["start"]),
            axis=1
        )
    else:
        rd["date_str"] = rd["start"].apply(lambda x: pd.Timestamp(x).strftime("%Y-%m-%d"))

    rd = rd[rd["date_str"].notna()].copy()

    daily = (
        rd.groupby(["collab_id", "date_str"])
        .agg(
            day_first_rda_start=("start", "min"),
            day_last_rda_end=("end", "max"),
        )
        .reset_index()
        .sort_values(["collab_id", "date_str"])
    )

    lookup = {}

    for cid, grp in daily.groupby("collab_id"):
        grp = grp.sort_values("date_str").reset_index(drop=True)

        for i, row in grp.iterrows():
            date_str = str(row["date_str"])

            cur_start = row["day_first_rda_start"]
            cur_end = row["day_last_rda_end"]

            prev_end = pd.NaT
            next_start = pd.NaT

            if i > 0:
                prev_end = grp.loc[i - 1, "day_last_rda_end"]

            if i < len(grp) - 1:
                next_start = grp.loc[i + 1, "day_first_rda_start"]

            prev_rest_min = np.nan
            next_rest_min = np.nan

            if pd.notna(prev_end) and pd.notna(cur_start):
                prev_rest_min = max(0.0, _duration_mins(prev_end, cur_start))

            if pd.notna(cur_end) and pd.notna(next_start):
                next_rest_min = max(0.0, _duration_mins(cur_end, next_start))

            lookup[(str(cid), date_str)] = {
                "cur_start": cur_start,
                "cur_end": cur_end,
                "prev_end": prev_end,
                "next_start": next_start,
                "prev_rest_min": prev_rest_min,
                "next_rest_min": next_rest_min,
            }

    return lookup


CMP_RDA_REST_LOOKUP = _cmp_build_rda_rest_lookup()


def _cmp_rest_texts_for_day(cid, date_str):
    info = CMP_RDA_REST_LOOKUP.get((str(cid), str(date_str)))

    if not info:
        return "REST BEFORE\n-", "REST AFTER\n-"

    prev_end = info["prev_end"]
    cur_start = info["cur_start"]
    cur_end = info["cur_end"]
    next_start = info["next_start"]

    prev_rest_min = info["prev_rest_min"]
    next_rest_min = info["next_rest_min"]

    if pd.notna(prev_rest_min):
        prev_duration = _fmt_hours(pd.Timedelta(minutes=float(prev_rest_min)))
        prev_text = (
            "REST BEFORE\n"
            f"{_cmp_fmt_dt_short(prev_end)} → {_cmp_fmt_dt_short(cur_start)}\n"
            f"{prev_duration}"
        )
    else:
        prev_text = (
            "REST BEFORE\n"
            "No previous RDA day"
        )

    if pd.notna(next_rest_min):
        next_duration = _fmt_hours(pd.Timedelta(minutes=float(next_rest_min)))
        next_text = (
            "REST AFTER\n"
            f"{_cmp_fmt_dt_short(cur_end)} → {_cmp_fmt_dt_short(next_start)}\n"
            f"{next_duration}"
        )
    else:
        next_text = (
            "REST AFTER\n"
            "No next RDA day"
        )

    return prev_text, next_text


# ============================================================
# WF all trips in the right-side label area
# ============================================================

def _cmp_wf_all_text_from_events(day_events):
    """
    Builds right-side label text showing ALL Webfleet trips of the day,
    numbered in chronological order.

    Each trip shows:
      number, start time, end time, duration, km if available.
    """
    wf_sub = day_events[day_events["kind"] == "WF"].copy()

    if wf_sub.empty:
        return "ALL WF TRIPS\n-"

    wf_sub = wf_sub.sort_values(["left", "right"]).reset_index(drop=True)

    lines = []

    for _, row in wf_sub.iterrows():
        idx = row.get("wf_index", np.nan)

        idx_txt = f"{int(idx):02d}" if pd.notna(idx) else "--"

        s = _fmt_hhmm(row["left"])
        e = _fmt_hhmm(row["right"])

        mins = _duration_mins(row["left"], row["right"])
        mins_txt = f"{int(round(mins))}m" if pd.notna(mins) else "-"

        km_txt = ""
        if "km_label" in row.index and str(row.get("km_label", "")).strip():
            km_txt = f" | {str(row.get('km_label')).strip()}"

        lines.append(f"{idx_txt}. {s} → {e} ({mins_txt}){km_txt}")

    return "ALL WF TRIPS\n" + "\n".join(lines)


# ============================================================
# Build event rows for the comparison chart
# ============================================================

def _cmp_build_event_rows():
    rows = []

    # -------------------------
    # WF lane
    # -------------------------
    wf_local = wf.copy() if "wf" in globals() else pd.DataFrame()

    if not wf_local.empty:
        wf_local = wf_local.dropna(subset=["collab_id", "start", "end"]).copy()
        wf_local["collab_id"] = wf_local["collab_id"].astype(str)
        wf_local = wf_local.sort_values(["collab_id", "start", "end"])

        wf_stack = {}

        for rr in wf_local.itertuples(index=False):
            s = _to_local_naive(rr.start)
            e = _to_local_naive(rr.end)

            if pd.isna(s) or pd.isna(e) or e <= s:
                continue

            cid = str(rr.collab_id)
            date_str = s.strftime("%Y-%m-%d")
            day_key = (cid, date_str)

            stack_idx = wf_stack.get(day_key, 0)
            wf_stack[day_key] = stack_idx + 1

            wf_index = stack_idx + 1

            km_label_y = -0.38 if (stack_idx % 2 == 0) else -0.48
            is_suspect = float(getattr(rr, "suspect_private_km", 0.0) or 0.0) > 0

            try:
                tripmode_int = int(rr.tripmode)
            except Exception:
                tripmode_int = -1

            fill_c = (
                "#d62728"
                if is_suspect
                else {1: "#6c757d", 2: "#ff7f0e", 3: "#1f77b4"}.get(tripmode_int, "#999999")
            )

            line_c = "#7f0000" if is_suspect else "#202020"
            km_txt = f"{float(rr.km):.1f}km" if hasattr(rr, "km") and pd.notna(rr.km) else ""

            rows.append({
                "collab_id": cid,
                "collab_label": CMP_COLLAB_LABELS.get(cid, cid),
                "date_str": date_str,
                "kind": "WF",
                "y": LANE_Y_CMP["WF"],
                "left": s,
                "right": e,
                "mid": s + (e - s) / 2,
                "height": 0.34,
                "fill_color": fill_c,
                "line_color": line_c,
                "label_text": f"{int(round(_duration_mins(s, e)))}m",
                "label_y": -0.20,
                "label_color": "#222222",
                "km_label": km_txt,
                "km_label_y": km_label_y,
                "km_label_color": "#d62728" if is_suspect else "#111111",
                "wf_index": wf_index,
            })

    # -------------------------
    # Original RDA lane
    # -------------------------
    rda_orig_local = rda.copy() if "rda" in globals() else pd.DataFrame()

    if not rda_orig_local.empty:
        rda_orig_local = rda_orig_local.dropna(subset=["collab_id", "start", "end"]).copy()
        rda_orig_local["collab_id"] = rda_orig_local["collab_id"].astype(str)

        for rr in rda_orig_local.itertuples(index=False):
            s = _to_local_naive(rr.start)
            e = _to_local_naive(rr.end)

            if pd.isna(s) or pd.isna(e) or e <= s:
                continue

            cid = str(rr.collab_id)
            code = getattr(rr, "prestation_code", np.nan)
            is_61010 = str(code) == str(PRESTATION_61010_CODE)

            # IMPORTANT:
            # Use RDA jour as the page date when available.
            # This prevents RDA Original from disappearing because of date mismatch.
            jour_value = getattr(rr, "jour", pd.NaT)
            date_str = _cmp_date_str(jour_value, fallback_ts=s)

            if not date_str:
                continue

            rows.append({
                "collab_id": cid,
                "collab_label": CMP_COLLAB_LABELS.get(cid, cid),
                "date_str": date_str,
                "kind": "RDA_ORIG",
                "y": LANE_Y_CMP["RDA_ORIG"],
                "left": s,
                "right": e,
                "mid": s + (e - s) / 2,
                "height": 0.34,
                "fill_color": _cmp_rda_orig_color(code),
                "line_color": "#202020",
                "label_text": f"{int(round(_duration_mins(s, e)))}m",
                "label_y": 0.80,
                "label_color": "#b30000" if is_61010 else "#111111",
                "km_label": "",
                "km_label_y": np.nan,
                "km_label_color": "#111111",
                "wf_index": np.nan,
            })

    # -------------------------
    # Planning lane
    # -------------------------
    planning_local = planning.copy() if "planning" in globals() else pd.DataFrame()

    if not planning_local.empty:
        planning_local = planning_local.dropna(subset=["collab_id", "start", "end"]).copy()
        planning_local["collab_id"] = planning_local["collab_id"].astype(str)

        for rr in planning_local.itertuples(index=False):
            s = _to_local_naive(rr.start)
            e = _to_local_naive(rr.end)

            if pd.isna(s) or pd.isna(e) or e <= s:
                continue

            cid = str(rr.collab_id)
            rr_dict = rr._asdict()

            # IMPORTANT:
            # Prefer planning date_only, then date, then fallback to start timestamp.
            plan_date_value = pd.NaT

            if "date_only" in rr_dict:
                plan_date_value = rr_dict.get("date_only", pd.NaT)
            elif "date" in rr_dict:
                plan_date_value = rr_dict.get("date", pd.NaT)

            date_str = _cmp_date_str(plan_date_value, fallback_ts=s)

            if not date_str:
                continue

            rows.append({
                "collab_id": cid,
                "collab_label": CMP_COLLAB_LABELS.get(cid, cid),
                "date_str": date_str,
                "kind": "Planning",
                "y": LANE_Y_CMP["Planning"],
                "left": s,
                "right": e,
                "mid": s + (e - s) / 2,
                "height": 0.34,
                "fill_color": _cmp_planning_color(rr_dict),
                "line_color": "#4A3B33",
                "label_text": f"{int(round(_duration_mins(s, e)))}m",
                "label_y": 1.80,
                "label_color": "#111111",
                "km_label": "",
                "km_label_y": np.nan,
                "km_label_color": "#111111",
                "wf_index": np.nan,
            })

    return pd.DataFrame(rows)


events_cmp = _cmp_build_event_rows()

if events_cmp.empty:
    raise RuntimeError(
        "events_cmp is empty. Make sure wf, rda and planning exist, "
        "and that the prior cells ran successfully."
    )

events_cmp = events_cmp.sort_values(
    ["collab_label", "date_str", "y", "left", "right"]
).reset_index(drop=True)

print("✅ Event rows by lane:", events_cmp["kind"].value_counts().to_dict())
print("✅ Unique collabs in PDF events:", events_cmp["collab_id"].nunique())
print("✅ Unique dates in PDF events:", events_cmp["date_str"].nunique())


# ============================================================
# Drawing helpers
# ============================================================

def _cmp_draw_global_markers(ax, day_events):
    """
    Draw dashed start/end and hourly markers based on ORIGINAL RDA span.
    """
    rda_orig_sub = day_events[day_events["kind"] == "RDA_ORIG"].copy()

    if rda_orig_sub.empty:
        return

    rs = _to_local_naive(rda_orig_sub["left"].min())
    re_ = _to_local_naive(rda_orig_sub["right"].max())

    if pd.isna(rs) or pd.isna(re_) or re_ <= rs:
        return

    ymin_full = -0.36
    ymax_full = 2.36

    for x, lab in [
        (rs, f"Start {rs.strftime('%H:%M')}"),
        (re_, f"End {re_.strftime('%H:%M')}")
    ]:
        xx = mdates.date2num(x)

        ax.vlines(
            xx,
            ymin=ymin_full,
            ymax=ymax_full,
            colors="#555555",
            linestyles="--",
            linewidth=2.0,
            alpha=0.90,
            zorder=1
        )

        ax.text(
            xx,
            2.52,
            lab,
            ha="center",
            va="bottom",
            fontsize=11.5,
            color="#555555",
            zorder=6,
            clip_on=False
        )

    span_hours = max(0.0, (re_ - rs).total_seconds() / 3600.0)
    total_hours = int(math.floor(span_hours)) + 1

    for h in range(1, total_hours + 1):
        x = rs + pd.Timedelta(hours=h)
        xx = mdates.date2num(x)

        ax.vlines(
            xx,
            ymin=ymin_full,
            ymax=ymax_full,
            colors="#9a9a9a",
            linestyles=":",
            linewidth=1.2,
            alpha=0.80,
            zorder=1
        )

        ax.text(
            xx,
            2.45,
            f"{h}h",
            ha="center",
            va="bottom",
            fontsize=10,
            color="#6f6f6f",
            zorder=6,
            clip_on=False
        )


def _cmp_draw_day(ax, day_events):
    for _, rr in day_events.iterrows():
        left = _to_local_naive(rr["left"])
        right = _to_local_naive(rr["right"])

        if pd.isna(left) or pd.isna(right) or right <= left:
            continue

        y = float(rr["y"])
        h = float(rr["height"])

        ax.broken_barh(
            [(mdates.date2num(left), mdates.date2num(right) - mdates.date2num(left))],
            (y - h / 2.0, h),
            facecolors=rr["fill_color"],
            edgecolors=rr["line_color"],
            linewidth=1.0,
            alpha=0.95,
            zorder=3,
        )

        mid = _to_local_naive(rr["mid"])
        lab = rr["label_text"]

        if pd.notna(mid) and str(lab).strip():
            ax.text(
                mdates.date2num(mid),
                rr["label_y"],
                lab,
                ha="center",
                va="center",
                fontsize=8.5,
                color=rr["label_color"],
                zorder=6,
                clip_on=False,
                bbox=dict(
                    facecolor="white",
                    edgecolor="none",
                    alpha=0.55,
                    pad=0.18
                ),
            )

        # Number on WF trip block, matching ALL WF TRIPS list
        if str(rr["kind"]) == "WF":
            wf_idx = rr.get("wf_index", np.nan)

            if pd.notna(mid) and pd.notna(wf_idx):
                ax.text(
                    mdates.date2num(mid),
                    float(rr["y"]) + 0.26,
                    f"{int(wf_idx)}",
                    ha="center",
                    va="center",
                    fontsize=8.3,
                    color="#000000",
                    fontweight="bold",
                    zorder=7,
                    clip_on=False,
                    bbox=dict(
                        facecolor="white",
                        edgecolor="#222222",
                        alpha=0.88,
                        boxstyle="round,pad=0.18"
                    ),
                )

        km_lab = str(rr.get("km_label", "") or "").strip()
        km_y = rr.get("km_label_y", np.nan)

        if pd.notna(mid) and km_lab and pd.notna(km_y):
            ax.text(
                mdates.date2num(mid),
                km_y,
                km_lab,
                ha="center",
                va="center",
                fontsize=8.2,
                color=rr["km_label_color"],
                zorder=6,
                clip_on=False,
                bbox=dict(
                    facecolor="white",
                    edgecolor="none",
                    alpha=0.55,
                    pad=0.15
                ),
            )


# ============================================================
# Build one PDF page
# ============================================================

def _cmp_build_day_page(collab_label, cid, date_str):
    # IMPORTANT:
    # Match by collab_id + date_str, not by collab_label.
    # This avoids label mismatch causing RDA or Planning to disappear.
    day_events = events_cmp[
        (events_cmp["collab_id"].astype(str) == str(cid)) &
        (events_cmp["date_str"].astype(str) == str(date_str))
    ].copy()

    if day_events.empty:
        return None

    day_events = day_events.sort_values(["y", "left", "right"]).reset_index(drop=True)

    left_bound, right_bound = _cmp_day_bounds(day_events, date_str, min_hours=16)

    wf_s, wf_e = _cmp_lane_span(day_events, "WF")
    ro_s, ro_e = _cmp_lane_span(day_events, "RDA_ORIG")
    pl_s, pl_e = _cmp_lane_span(day_events, "Planning")

    km_stats = _cmp_month_km_stats(cid, date_str)

    km_month_line = (
        f"Month KM ({km_stats['month_label']})"
        f"    |    Private: {km_stats['km_private']:.1f}km"
        f"    |    Suspect: {km_stats['km_suspect_private']:.1f}km"
        f"    |    Private+Suspect: {km_stats['km_private_plus_suspect']:.1f}km"
        f"    |    Dom-travail: {km_stats['km_commute']:.1f}km"
        f"    |    Professionnel: {km_stats['km_professional']:.1f}km"
        f"    |    Total: {km_stats['km_total']:.1f}km"
    )

    wf_all_text = _cmp_wf_all_text_from_events(day_events)

    wf_trip_count = int((day_events["kind"] == "WF").sum())

    if wf_trip_count <= 12:
        wf_text_fontsize = 8.4
    elif wf_trip_count <= 25:
        wf_text_fontsize = 7.3
    elif wf_trip_count <= 40:
        wf_text_fontsize = 6.4
    elif wf_trip_count <= 60:
        wf_text_fontsize = 5.5
    else:
        wf_text_fontsize = 4.8

    prev_rest_text, next_rest_text = _cmp_rest_texts_for_day(cid, date_str)

    subtitle = (
        f"Frame: {_fmt_hhmm(left_bound)} → {_fmt_hhmm(right_bound)}"
        f"    |    WF span: {_fmt_span(wf_s, wf_e)}"
        f"    |    RDA original: {_fmt_span(ro_s, ro_e)}"
        f"    |    Planning: {_fmt_span(pl_s, pl_e)}"
    )

    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H), constrained_layout=False)

    _cmp_draw_global_markers(ax, day_events)
    _cmp_draw_day(ax, day_events)

    ax.set_xlim(mdates.date2num(left_bound), mdates.date2num(right_bound))
    ax.set_ylim(-0.52, 2.72)
    ax.margins(x=0, y=0)

    ax.set_yticks([0.0, 1.0, 2.0])
    ax.set_yticklabels(
        ["WF Trips", "RDA Original", "Planning"],
        fontsize=11.5
    )

    span_hours = max(1.0, (right_bound - left_bound).total_seconds() / 3600.0)
    major_interval = 1 if span_hours <= 14 else 2 if span_hours <= 22 else 3

    ax.xaxis.set_major_locator(mdates.HourLocator(interval=major_interval))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.tick_params(axis="x", labelsize=11)

    ax.grid(axis="x", linestyle=":", linewidth=0.8, alpha=0.45)
    ax.grid(axis="y", visible=False)

    for sp in ax.spines.values():
        sp.set_linewidth(0.8)
        sp.set_alpha(0.70)

    fig.suptitle(
        f"{collab_label} | {date_str}",
        fontsize=14,
        y=0.985,
        fontweight="bold"
    )

    fig.text(
        0.5,
        0.948,
        km_month_line,
        ha="center",
        va="center",
        fontsize=10.5,
        fontweight="bold",
        bbox=dict(
            facecolor="#f2f2f2",
            edgecolor="#d0d0d0",
            boxstyle="round,pad=0.30"
        )
    )

    fig.text(
        0.5,
        0.915,
        subtitle,
        ha="center",
        va="center",
        fontsize=9.2,
        color="#333333"
    )

    # Left top label: rest from previous RDA day to this RDA day
    fig.text(
        0.105,
        0.875,
        prev_rest_text,
        ha="left",
        va="top",
        fontsize=8.7,
        family="monospace",
        color="#222222",
        bbox=dict(
            facecolor="#ffffff",
            edgecolor="#cfcfcf",
            boxstyle="round,pad=0.35",
            alpha=0.96
        )
    )

    # Right top label: rest from this RDA day to next RDA day
    fig.text(
        0.775,
        0.875,
        next_rest_text,
        ha="right",
        va="top",
        fontsize=8.7,
        family="monospace",
        color="#222222",
        bbox=dict(
            facecolor="#ffffff",
            edgecolor="#cfcfcf",
            boxstyle="round,pad=0.35",
            alpha=0.96
        )
    )

    # Right-side ALL WF trips label area
    fig.text(
        0.805,
        0.775,
        wf_all_text,
        ha="left",
        va="top",
        fontsize=wf_text_fontsize,
        family="monospace",
        color="#222222",
        bbox=dict(
            facecolor="#ffffff",
            edgecolor="#cfcfcf",
            boxstyle="round,pad=0.45",
            alpha=0.96
        )
    )

    # Leave space on right for WF trip label box
    fig.subplots_adjust(left=0.10, right=0.775, top=0.79, bottom=0.12)

    return fig


# ============================================================
# Build all PDFs — one PDF per collaborator
# ============================================================

pairs_cmp = (
    events_cmp[["collab_id", "collab_label", "date_str"]]
    .drop_duplicates()
    .copy()
)

pairs_cmp["collab_id"] = pairs_cmp["collab_id"].astype(str)
pairs_cmp["date_str"] = pairs_cmp["date_str"].astype(str)

pairs_cmp = pairs_cmp.sort_values(
    ["collab_label", "date_str"]
).reset_index(drop=True)

manifest_rows = []

for collab_label, grp in pairs_cmp.groupby("collab_label", sort=True):
    grp = grp.sort_values("date_str").reset_index(drop=True)

    cid_values = grp["collab_id"].dropna().astype(str).unique().tolist()
    cid = cid_values[0] if cid_values else "unknown"

    pdf_name = f"{_safe_filename(collab_label)}__collab_{cid}__wf_rda_planning.pdf"
    pdf_path = COMPARE_PDF_DIR / pdf_name

    pages_written = 0

    with PdfPages(pdf_path) as pdf:
        for _, rr in grp.iterrows():
            fig = _cmp_build_day_page(
                rr["collab_label"],
                str(rr["collab_id"]),
                str(rr["date_str"])
            )

            if fig is None:
                continue

            pdf.savefig(fig, dpi=220, bbox_inches="tight")
            plt.close(fig)

            pages_written += 1

    manifest_rows.append({
        "collab_id": cid,
        "collab_label": collab_label,
        "pdf_path": str(pdf_path),
        "pages_written": pages_written,
    })

    print(f"✅ {collab_label} -> {pages_written} pages -> {pdf_path}")

manifest_cmp = (
    pd.DataFrame(manifest_rows)
    .sort_values(["collab_label"])
    .reset_index(drop=True)
)

manifest_cmp_path = COMPARE_PDF_DIR / "pdf_manifest_comparison.csv"
manifest_cmp.to_csv(manifest_cmp_path, index=False, encoding="utf-8")

print("\n✅ Done.")
print("Comparison PDF folder:", COMPARE_PDF_DIR)
print("Manifest:", manifest_cmp_path)
print("✅ Modified RDA completely removed.")
print("✅ One PDF created per collaborator.")
print("✅ RDA Original uses RDA jour for page matching.")
print("✅ Planning uses date_only/date/start fallback for page matching.")
print("✅ ALL WF trips are listed in order and numbered on the WF lane.")

✅ Event rows by lane: {'RDA_ORIG': 11696, 'WF': 6063, 'Planning': 5483}
✅ Unique collabs in PDF events: 84
✅ Unique dates in PDF events: 31
✅ Abastanotti Anita (ID collab-collab-0226, SA:876, WF:309) -> 10 pages -> /content/drive/MyDrive/sarl year 2025/Webfleet_RDA_Audit_20260512_101940/05_Plots_PDF_Comparison/Abastanotti_Anita__ID_collab-collab-0226__SA_876__WF_309___collab_collab-0226__wf_rda_planning.pdf
✅ Angeles Elguera Giomara Lucila (ID collab-collab-0017, SA:6641, WF:-) -> 1 pages -> /content/drive/MyDrive/sarl year 2025/Webfleet_RDA_Audit_20260512_101940/05_Plots_PDF_Comparison/Angeles_Elguera_Giomara_Lucila__ID_collab-collab-0017__SA_6641__WF_-___collab_collab-0017__wf_rda_planning.pdf
✅ Armici Raffaele (ID collab-collab-0228, SA:1027, WF:362) -> 17 pages -> /content/drive/MyDrive/sarl year 2025/Webfleet_RDA_Audit_20260512_101940/05_Plots_PDF_Comparison/Armici_Raffaele__ID_collab-collab-0228__SA_1027__WF_362___collab_collab-0228__wf_rda_planning.pdf
✅ Bakowska Maria (ID colla

In [ ]:
import pandas as pd

# Non-mutating check: which collaborator/days have all lanes, planning only, or RDA/WF only?
check_df = (
    events_cmp.groupby(['collab_id', 'date_str'])['kind']
    .apply(lambda x: sorted(set(x)))
    .reset_index(name='lanes')
)
check_df['has_planning'] = check_df['lanes'].apply(lambda xs: 'Planning' in xs)
check_df['has_rda_or_wf'] = check_df['lanes'].apply(lambda xs: ('RDA_ORIG' in xs) or ('WF' in xs))
check_df['status'] = check_df.apply(
    lambda r: 'all_or_combined' if r['has_planning'] and r['has_rda_or_wf'] else (
        'planning_only' if r['has_planning'] else 'rda_wf_only'
    ),
    axis=1,
)

print('Planning/RDA/WF day alignment:')
display(check_df['status'].value_counts().rename_axis('status').reset_index(name='days'))

print('Days with RDA/WF but no planning:')
display(check_df[check_df['status'].eq('rda_wf_only')].head(30))

print('Days with planning but no RDA/WF:')
display(check_df[check_df['status'].eq('planning_only')].head(30))

if 'planning_dropped_debug' in globals() and not planning_dropped_debug.empty:
    print('Dropped planning rows by reason:')
    display(planning_dropped_debug['drop_reason'].value_counts().rename_axis('drop_reason').reset_index(name='rows'))


Planning/RDA/WF day alignment:


,status,days
0,all_or_combined,852
1,rda_wf_only,332
2,planning_only,112


Days with RDA/WF but no planning:


,collab_id,date_str,lanes,has_planning,has_rda_or_wf,status
3,collab-0030,2026-04-01,[RDA_ORIG],False,True,rda_wf_only
32,collab-0033,2026-04-30,[RDA_ORIG],False,True,rda_wf_only
70,collab-0057,2026-04-02,[WF],False,True,rda_wf_only
75,collab-0057,2026-04-07,[WF],False,True,rda_wf_only
76,collab-0057,2026-04-08,[WF],False,True,rda_wf_only
77,collab-0057,2026-04-10,[WF],False,True,rda_wf_only
82,collab-0057,2026-04-20,[WF],False,True,rda_wf_only
83,collab-0057,2026-04-21,[WF],False,True,rda_wf_only
84,collab-0057,2026-04-22,[WF],False,True,rda_wf_only
85,collab-0057,2026-04-24,[WF],False,True,rda_wf_only


Days with planning but no RDA/WF:


,collab_id,date_str,lanes,has_planning,has_rda_or_wf,status
33,collab-0039,2026-04-01,[Planning],True,False,planning_only
34,collab-0039,2026-04-04,[Planning],True,False,planning_only
35,collab-0039,2026-04-05,[Planning],True,False,planning_only
36,collab-0039,2026-04-08,[Planning],True,False,planning_only
37,collab-0039,2026-04-11,[Planning],True,False,planning_only
38,collab-0039,2026-04-12,[Planning],True,False,planning_only
39,collab-0039,2026-04-15,[Planning],True,False,planning_only
40,collab-0039,2026-04-18,[Planning],True,False,planning_only
41,collab-0039,2026-04-19,[Planning],True,False,planning_only
42,collab-0039,2026-04-22,[Planning],True,False,planning_only


Dropped planning rows by reason:


,drop_reason,rows
0,unmapped_emp_nr,1283


In [ ]:
# =========================
# Deprecated temporary planning date swap patch
# =========================
print("This old temporary patch is disabled. Planning dates are now parsed correctly in Cell 5.")
print("Run Cell 5 and then the Comparison PDF export cell to rebuild the PDFs.")


This old temporary patch is disabled. Planning dates are now parsed correctly in Cell 5.
Run Cell 5 and then the Comparison PDF export cell to rebuild the PDFs.


In [ ]:
# =========================
# New Cell C — Modified RDA summary DISABLED
# =========================
print("✅ Modified/Adjusted RDA summary removed.")
print("✅ No modified RDA file.")
print("✅ No modified RDA axis.")
print("✅ No modified RDA comparison stats.")

✅ Modified/Adjusted RDA summary removed.
✅ No modified RDA file.
✅ No modified RDA axis.
✅ No modified RDA comparison stats.


In [ ]:
# @title
# =========================
# Check for Unmapped RDA Collaborators
# =========================
import pandas as pd
from IPython.display import display

print("Checking for RDA rows that could not be mapped to a collab_id...")

# Find rows where the collab_id mapping failed (is NaN)
unmapped_rda = rda[rda['collab_id'].isna()]

if unmapped_rda.empty:
    print("\n✅ All RDA rows were successfully mapped to a collab_id!")
else:
    print(f"\n⚠️ Found {len(unmapped_rda)} RDA rows with unmapped collaborators.")

    # Group by the SARL number and Name to see who exactly is missing
    unmapped_summary = unmapped_rda[['collab_no_sarl', 'collab_name']].drop_duplicates()

    print("\nList of unmapped collaborators in RDA:")
    display(unmapped_summary)


Checking for RDA rows that could not be mapped to a collab_id...

✅ All RDA rows were successfully mapped to a collab_id!


In [ ]:
# @title
# =========================
# Enhanced Check: Unrecognized RDA Collaborators
# =========================
import pandas as pd
from IPython.display import display

# Filter for RDA rows where the master collab_id is missing (NaN)
unmapped_rda = rda[rda['collab_id'].isna()].copy()

if unmapped_rda.empty:
    print("\u2705 All RDA records successfully matched to a known master collab_id!")
else:
    print(f"\u26a0\ufe0f Found {len(unmapped_rda)} RDA rows with unmapped collaborators.\n")

    # Group by the SARL number and Name, and count the occurrences
    unmapped_summary = (
        unmapped_rda[['collab_no_sarl', 'collab_name']]
        .value_counts(dropna=False)
        .reset_index(name='affected_rda_rows')
        .sort_values('affected_rda_rows', ascending=False)
    )

    print("Unrecognized Collaborators Summary:")
    display(unmapped_summary)


✅ All RDA records successfully matched to a known master collab_id!
